# E13 RC1 — Qwen3.8-Flash-Next on 4xL4: root cause of the layer-2 MoE NaN (SGLang 0.5.21) + vLLM 0.32 smoke
NANPROBE rev 2 put the first NaN in model.layers.2.mlp.experts (GPTQMarlinMoE, finite float inputs) on all ranks. This notebook (RC1 fix list, 3 GPT audits): arm 1 SGLang eager with hooks on layers 0-3, expert weight scans after the repack and after forward 0, every MoE input incl. top-k ids, MoE fixtures of layers 1-2 recomputed on the CPU from the checkpoint's GPTQ weights (Marlin vs reference), the runtime gptq_moe source sha, and the count prompt under three PLE modes (orig / sync / skip) in the same server; arm 2 Flash-Next on vLLM 0.32.0 (native qwen4_exp, TP4 + EP, eager, PLE in pinned host memory): count, 2+2 chat, 4 streams; arm 3 (time permitting) layer 2 on the non-Marlin GPTQ path. No Duck; the submission is CELL 0's fallback.

# E13 — Duck REPL agent for ARC-AGI-2 on SGLang 0.5.21 + Qwen3.8-Flash-Next (Intel W4A16, BF16 PLE) (SMOKE1)
The E12 Duck notebook unchanged except the engine (CELL 5) and its inputs/limits in CELL 1. Engine = Franzen's ARC-AGI-3 milestone-2 launch with the 4xL4 deltas (PLAN rev 4.1 simple path): 0.5.21, TP4+EP4, Triton GDN, arch 8.9, classic FA2 for QSA decode, Franzen patch #3; memory items NEXTN off, chunk 2048, mem 0.93, 4 running; Duck streams = min(4, KV // 60k).

In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "SMOKE1").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "EVAL14":    {"source": "eval", "task_list": ["135a2760", "136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "3dc255db", "4c3d4a41", "4c416de3", "4c7dc4dd", "62593bfd", "64efde09", "65b59efc", "6e4f6532"],
                  "time_budget_sec": 2.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 step 3.5: GPU smoke, 1 fast task (2ba387bc: 5,355 thinking tokens in E10 ORDER2), 2 branches, 15 min solving
    "SMOKE1":    {"source": "eval", "task_list": ["2ba387bc"], "time_budget_sec": 120 * 60, "admit_frac": 0.90,
                  "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60,
                  "admit_margin_sec": 2 * 60, "solve_window_sec": 15 * 60},
    # E12 step 4: the E11-05 EVAL30 tasks (same seed sample), 4 h
    "EVAL30":    {"source": "eval", "n_tasks": 30, "time_budget_sec": 4.0 * 3600,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # E11 rev 3: 60,000 = S1-60K (LB 34.58); user floor 60k   # S49: LB test of 49,152 (S1: 60,000; R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
E11_THINK_FLOOR = 60000      # E11 rev 3 (user 2026-10-18): never below 60k (budget and admission guard)
assert THINK_BUDGET >= E11_THINK_FLOOR, (THINK_BUDGET, E11_THINK_FLOOR)
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
# ---- E11-05: engine = SGLang 0.5.21 + RedHat INT4 (W4A16 g128) + DFlash2 drafter = the E11-04 W1 launch (2026-10-18).
# MODEL_PATH (FP8) stays: its tokenizer + chat template render every prompt exactly as in S49, and it is the vLLM fallback.
ENGINE_BACKEND = os.environ.get("E11_ENGINE", "sglang").strip().lower()       # sglang (vLLM FP8 fallback) | vllm (= S49)
E11_ALLOW_VLLM_FALLBACK = os.environ.get("E11_VLLM_FALLBACK", "1") == "1"
SGL_MODEL_PATH = Path(os.environ.get("E11_INT4", "/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1"))   # E13
SGL_DRAFT_PATH = Path(os.environ.get("E11_DFLASH", "/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1"))   # E13: precache only (NEXTN off)
SGL_WHEELHOUSE = Path(os.environ.get("E11_SGL_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse"))


def _e11_discover(path, marker, pred):
    """The given path if it holds `marker`, else the first /kaggle/input folder whose `marker` json satisfies pred."""
    if (path / marker).is_file():
        return path
    root = Path("/kaggle/input")
    for p in (sorted(root.rglob(marker)) if root.exists() else []):
        try:
            if pred(p.parent, _load_json(p)):
                return p.parent
        except Exception:
            continue
    return path


SGL_MODEL_PATH = _e11_discover(SGL_MODEL_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and (c.get("quantization_config") or c.get("text_config", {}).get("quantization_config") or {}).get("quant_method") == "auto-round")   # E13
SGL_DRAFT_PATH = _e11_discover(SGL_DRAFT_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and "mtp_routed_experts" in ((c.get("quantization_config") or {}).get("config_groups") or {}))   # E13: Franzen's draft test
SGL_WHEELHOUSE = _e11_discover(SGL_WHEELHOUSE, "BUILD_INFO.json", lambda d, c: isinstance(c, dict)
                               and c.get("sglang") == "0.5.21" and (d / "requirements.lock").is_file())
SGL_CONTEXT_LEN = 139264            # = the E11-04 W1 launch; every request stays inside MAX_MODEL_LEN (phase1_cap)
SGL_MAX_RUNNING = 4                 # E11-04 W1: 12 really running (mamba cache 56 / 4 state slots = 14 >= 12)
SGL_SERVER_MAX_RUNNING = 4          # rev 3c: server admission limit = client 12 + 1. SGLang 0.5.21 counts the row of a request in
                                    # chunked prefill twice (scheduler.py:3971-3976) and latches batch_is_full (3846-3848) until a
                                    # running request ends -> EVAL30 ran 11 + 1 queued (2,358 of 2,509 decode lines). Upstream fix
                                    # PR #44401 (after 0.5.21); +1 is the documented workaround (PR #30295). CUDA graphs stay at 12.
SGL_LAUNCH_TIMEOUT_SEC = 30 * 60    # first launch includes JIT + CUDA graphs: 567 s in E11-04
SGL_RELAUNCH_TIMEOUT_SEC = 20 * 60  # a diagnosed relaunch (E11-04: about 75 s once the JIT cache exists)
SGL_GIVE_UP_AT = min(NOTEBOOK_T0 + 75 * 60, ADMIT_DEADLINE - 5 * 60)   # no SGLang launch after this -> vLLM fallback
SGL_STREAM_IDLE_SEC = 1800          # a socket read silent for longer closes that request as 'error'
# rev 2 (GPT audit 2026-10-18, verified): capacity gate from the server log, fail-closed on missing evidence
SGL_MIN_RUNNING = 8                 # below this the SGLang engine is refused (-> S49 vLLM)
SGL_MIN_KV = 600_000                # KV tokens; E11-04 W1 had 881,336
SGL_KV_PER_REQUEST = 70_000         # concurrency <= KV // this (881,336 -> 12); a long Poetiq unit is prompt + 49k + 8k
SGL_MIN_FREE_GB = 1.0               # GPU headroom after start-up (E11-04 W1: 2.22 GB)
# rev 2: drain earlier. Aborted long prefixes were not cached in E11-04 (30k prefix: 0 cached, 24.5 s), so 12 phase-2
# continuations re-prefill up to ~0.7M tokens; 12 min instead of 8 min before the hard stop (SGLang only)
SGL_DRAIN_MARGIN_SEC = 8 * 60      # E11-06: back to 8 min (EVAL30 drain <= 83 s)
# rev 2: KV gate active on SGLang (S49's floor 12 = CONCURRENCY 12 made it inert): from 10 running, admit only below 85 %
SGL_KV_GATE_MIN_ACTIVE = 10
SGL_KV_ADMIT_MAX = 0.85

CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e11"] = {k: (str(globals()[k]) if isinstance(globals()[k], Path) else globals()[k]) for k in (
    "ENGINE_BACKEND", "E11_ALLOW_VLLM_FALLBACK", "SGL_MODEL_PATH", "SGL_DRAFT_PATH", "SGL_WHEELHOUSE", "SGL_CONTEXT_LEN",
    "SGL_MAX_RUNNING", "SGL_SERVER_MAX_RUNNING", "SGL_LAUNCH_TIMEOUT_SEC", "SGL_RELAUNCH_TIMEOUT_SEC", "SGL_STREAM_IDLE_SEC", "SGL_MIN_RUNNING",
    "SGL_MIN_KV", "SGL_KV_PER_REQUEST", "SGL_MIN_FREE_GB", "SGL_DRAIN_MARGIN_SEC", "SGL_KV_GATE_MIN_ACTIVE", "SGL_KV_ADMIT_MAX")}
CONFIG["e11"]["give_up_after_sec"] = round(SGL_GIVE_UP_AT - NOTEBOOK_T0)
print("E11 engine:", json.dumps(CONFIG["e11"]))
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','2'))
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','0')=='1'
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}


# ---- E12: Duck REPL agent (Experiment 12, user option A 2026-10-19). Duck's own settings are in arc2_duck.DUCK_ENV
# (= dfranzen milestone-2 cell 4): 12,288 output per request, yield at 2,048 generated tokens, preserved thinking,
# 128 Ki context with drain to 58 Ki. Approved exception to the 60k-per-request floor (thinking accumulates in context).
E12_BRANCHES = int(os.environ.get("E12_BRANCHES", 2))                 # independent branches per task -> attempt_1/2
E12_STREAMS = int(os.environ.get("E12_STREAMS", 0))                   # Duck ARC3_MAX_ACTIVE_STREAMS; 0 = engine concurrency
E12_MAX_GEN_PER_BRANCH = int(os.environ.get("E12_MAX_GEN", 0)) or None   # Duck max_generated_tokens_per_game (milestone: none)
E12_SOLVE_WINDOW_SEC = P.get("solve_window_sec")                      # SMOKE1 only
E12_STOP_BEFORE_HARD_SEC = 180   # branches stop; outputs are written after it. rev 3 (GPT re-audit): a submit at the
                                 # deadline can still verify for <= 60 s, then CELL 6/7 write; 60 s left no margin
E12_MAMBA_TRACK_64 = os.environ.get("E12_MAMBA_TRACK_64", "0") == "1"  # Duck's --mamba-track-interval/--page-size 64 (measure)
# rev 2 (GPT audits, validated 2026-10-19): Duck request snapshots (the exact messages sent; large) for the smoke only,
# and a per-sandbox-process address-space limit (Duck sets CPU / file size / open files only)
E12_REQUEST_LOGS = os.environ.get("E12_REQUEST_LOGS", "1" if PROFILE == "SMOKE1" else "0") == "1"
E12_SANDBOX_MEM_MB = int(os.environ.get("E12_SANDBOX_MEM_MB", 4096))
CONFIG["e12"] = {k: globals()[k] for k in ("E12_BRANCHES", "E12_STREAMS", "E12_MAX_GEN_PER_BRANCH", "E12_SOLVE_WINDOW_SEC",
                                           "E12_STOP_BEFORE_HARD_SEC", "E12_MAMBA_TRACK_64", "E12_REQUEST_LOGS",
                                           "E12_SANDBOX_MEM_MB")}
CONFIG["e12"]["duck_commit"] = "10882e332d883af3e61616df8c8a17d03a328a75"
print("E12:", json.dumps(CONFIG["e12"]))

# ---- E13 (Flash-Next on 4xL4, PLAN rev 4.1 simple path): engine constants. Everything else in this notebook = E12.
E13_FA2_WHEEL = "flash_attn-2.8.3.post1+cu13torch2.13cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
E13_FA2_SHA = "74984f628eaa9b22de271c787d82c4ef16daffcfb9684286a7a15ff6ad41ae84"
# rev 6: the foysal copy first, then the hosen42 copy, then any directory under /kaggle/input holding the wheel (the name with
# or without '+': Kaggle drops it); the wheel's sha256 is checked before use either way
E13_FA2_DIRS = ([Path(os.environ["E13_FA2_DIR"])] if os.environ.get("E13_FA2_DIR") else []) + [
    Path("/kaggle/input/datasets/foysalemonshanto/flash-attn-2-8-3-post1-cu13-torch2-13-cp312"),
    Path("/kaggle/input/datasets/hosen42/flash-attn-2-8-3-post1-cu13-torch2-13-cp312")]
E13_FA2_DIR = next((d for d in E13_FA2_DIRS if d.is_dir()), E13_FA2_DIRS[0])
E13_PATCH3 = ("135e004f7613a59aa0765cbe3d779c9bb0c459f3a199b868358759e020aaeb5d",   # Franzen BEFORE_SHA (= 0.5.21 file)
              "47e65140e2480a8637b9ea5bb746ff77e9c09c01e448fd2056c0b485e4f3a10a")   # Franzen AFTER_SHA (= Pennyroyal wheel)
E13_MEM_FRACTION = "0.93"        # PLAN 3b: about 1.5 GiB outside the static pools (Franzen 0.96 on a 96 GB card)
E13_CHUNK = 2048                 # PLAN 3b: E3-proven on 4xL4 (Franzen 8192 on 96 GB)
E13_KV_PER_STREAM = 60000        # pre-run capacity rule: resident Duck contexts <= KV / 60k (E12 EVAL30 prompts p50 47.9k)
E13_TOPK_TEST = os.environ.get("E13_TOPK_TEST", "1") == "1"   # user: the fast_topk<512> correctness test always runs
E13_SKIP_CAT_PREFETCH = True      # rev 6b (audit B-H1): no CELL 2 background `cat` of 178 GB (it competes with the loader's cache)
CONFIG["e13"] = {k: globals()[k] for k in ("E13_FA2_SHA", "E13_MEM_FRACTION", "E13_CHUNK", "E13_KV_PER_STREAM", "E13_TOPK_TEST",
                                           "SGL_CONTEXT_LEN", "SGL_MAX_RUNNING")}
CONFIG["e13"]["fa2_dir"] = str(E13_FA2_DIR)

# ---- E13 LOADPROBE (user 2026-10-27): load-only, two launches with a GPU memory trace; no speed stage, no Duck
E13_LOADPROBE = True
E13_PROBE_ARMS = ("bf16ple", "fp8ple")          # 1 = rev 6b exactly (Franzen's BF16 PLE); 2 = the E3 v4.2.1 FP8 PLE merge
E13_INTEL_DIR = SGL_MODEL_PATH
_e13_ov = [Path(os.environ["E13_OVERLAY"])] if os.environ.get("E13_OVERLAY") else [
    Path("/kaggle/input/models/foysalemonshanto/qwen38-flash-next-fp8ple-overlay/pytorch/albucino-fp8ple/1")]
_e13_ovroot = Path("/kaggle/input/models/foysalemonshanto/qwen38-flash-next-fp8ple-overlay")
_e13_ov += sorted(_e13_ovroot.glob("*/*/*")) if _e13_ovroot.is_dir() else []
E13_OVERLAY_DIR = next((d for d in _e13_ov if (d / "config.json").is_file()), _e13_ov[0])
E13_OVERLAY_SHARDS = {"model-00001-of-00017.safetensors": 2899311784, "model-00003-of-00017.safetensors": 5370548192, "model-00004-of-00017.safetensors": 5370910416, "model-00005-of-00017.safetensors": 5370140704, "model-00006-of-00017.safetensors": 5370205184, "model-00007-of-00017.safetensors": 5370927288, "model-00008-of-00017.safetensors": 5344445352, "model-00009-of-00017.safetensors": 5370355648, "model-00010-of-00017.safetensors": 5370927288, "model-00011-of-00017.safetensors": 5324173744, "model-00012-of-00017.safetensors": 5370438520, "model-00013-of-00017.safetensors": 5370927288, "model-00014-of-00017.safetensors": 5342689856, "model-00015-of-00017.safetensors": 5050474736, "model-00017-of-00017.safetensors": 1284526608, "model-plefp8-00000.safetensors": 5200027199, "model-plefp8-00001.safetensors": 5200027202, "model-plefp8-00002.safetensors": 5200027198, "model-plefp8-00003.safetensors": 5200027189, "model-plefp8-00004.safetensors": 5200027190, "model-plefp8-00005.safetensors": 5200027190, "model-plefp8-00006.safetensors": 5200027190, "model-plefp8-00007.safetensors": 5200027190, "model-plefp8-00008.safetensors": 5200027190, "model-plefp8-00009.safetensors": 4400023163}                  # E3 v4.2.1 pinned sizes: 15 Intel main shards + 10 FP8 PLE shards
CONFIG["e13"]["loadprobe"] = {"arms": list(E13_PROBE_ARMS), "intel": str(E13_INTEL_DIR), "overlay": str(E13_OVERLAY_DIR)}

# ---- E13 NANPROBE (user 2026-10-30): the two Flash-Next blockers on 4xL4 in one session.
# Arm 1 nan_eager: BF16 PLE, gc fix, eager (no CUDA graphs) + the NaN / collapse locator (instance-forward wrapper, per
#   rank and forward: first non-finite module + origin vs propagated, layer [min, max], logits stats, QSA indexer ints);
#   count smoke, short chat only when the count passes; then a drain past the hook window and the count again (hooks off).
#   An OOM before ready -> one retry with gc after every module (nan_eager_gc0), and kv_lean is dropped.
# Arm 2 kv_lean: BF16 PLE, gc fix, graphs on, mamba cache 20 slots (= 4 running x 5 for extra_buffer + overlap: 8 slots
#   would cap running to 1) and ONLY mem fraction 0.93 -> 0.95 (LOADPROBE: mamba 0.29 GB, KV 0.54 GB, 1.05 GB free after
#   graphs at 0.93); capacity gate (running cap must stay 4) + 4 concurrent ~20k-token prompts (OOM / headroom).
# FP8 PLE is not an arm: LOADPROBE showed it changes host memory only (GPU tensors byte-identical, KV unchanged).
# After the arms (CPU only): a vLLM 0.32 registry check (our wheelhouse venv): is Flash-Next's architecture there?
E13_PROBE_ARMS = ("nan_eager", "kv_lean")
_e13_nan_env = {"E13_NANPROBE": "1", "E13_NAN_FORWARDS": "120", "E13_NAN_ABSMAX": "1e4", "E13_GC_MIB": "256"}
E13_ARM_CFG = {
    "nan_eager": {"model": "intel", "env": _e13_nan_env, "add": ["--disable-cuda-graph"], "nan": True, "chat_max_tokens": 256,
                  "rerun_count": True, "oom_retry": "nan_eager_gc0", "budget_sec": 30 * 60},
    "nan_eager_gc0": {"model": "intel", "env": {**_e13_nan_env, "E13_GC_MIB": "0"}, "add": ["--disable-cuda-graph"], "nan": True,
                      "chat_max_tokens": 256, "rerun_count": True, "budget_sec": 30 * 60},
    "kv_lean": {"model": "intel", "env": {"E13_GC_MIB": "256"}, "set": {"--mem-fraction-static": "0.95"},
                "chat_max_tokens": 256, "capacity_gate": True, "long_test": {"streams": 4, "tokens": 20000},
                "budget_sec": 30 * 60}}
E13_VLLM_CHECK = True
CONFIG["e13"]["nanprobe"] = {"arms": list(E13_PROBE_ARMS), "cfg": E13_ARM_CFG, "vllm_check": E13_VLLM_CHECK}

# ---- E13 RC1 (user 2026-10-30): root cause of the layer-2 MoE NaN (NANPROBE rev 2 + two GPT audits) and Flash-Next on
# vLLM 0.32. One notebook, three arms in order (each with its own deadline; a GPU-free gate between arms):
# Arm 1 rc1 (SGLang, eager, NANPROBE flags): hooks on layers 0-3 only; expert weight scan of layers 0-3 after the repack and
#   after forward 0 (non-finite scales + the local experts holding them, int checksums, expert maps, quant method); every
#   MoE input incl. the integer top-k ids (dtype, shape, stride, min / max); then the count prompt under three PLE modes
#   in the same server (flag file; cache flushed + a different prompt per mode): orig, sync (side stream synchronized),
#   skip (no prefetch, main-stream gather).
# Arm 2 vllm_fn (vLLM 0.32.0, native qwen4_exp): TP4 + EP, eager, 8k context, PLE in pinned host memory (--engram-config),
#   language model only; count + 2+2 chat + 4 x 256-token streams (tok/s); PLE / quant method / KV lines from the log.
# Arm 3 l2_wna16 (SGLang, only with >= 22 min left): arm 1 with layer 2's experts on the non-Marlin GPTQ path (MoeWNA16).
E13_PROBE_ARMS = ("rc1", "vllm_fn", "l2_wna16")
_e13_rc1_env = {"E13_NANPROBE": "1", "E13_NAN_FORWARDS": "240", "E13_NAN_ABSMAX": "1e4", "E13_GC_MIB": "256",
                "E13_NAN_ONLY": r"(^|\.)layers\.[0-3](\.|$)", "E13_RC1_SCAN": "1", "E13_RC1_LAYERS": "0,1,2,3",
                "E13_PLE_FLAG": os.environ.get("E13_PLE_FLAG", "/tmp/e13_ple_mode"), "E13_RC1_FIXTURE": "1", "E13_RC1_FIX_LAYERS": "1,2",
                "E13_RC1_FIX_FORWARDS": "2", "E13_RC1_FIX_TOKENS": "6",
                "E13_SYNC_RE": r"layers\.2\.mlp\.experts$"}   # RC1 audit extra B: device-wide sync around layer 2's MoE
E13_ARM_CFG = {
    "rc1": {"model": "intel", "env": _e13_rc1_env, "add": ["--disable-cuda-graph"], "nan": True, "chat_max_tokens": 0,
            "ple_modes": ["orig", "sync", "skip"], "budget_sec": 32 * 60},
    "vllm_fn": {"engine": "vllm", "budget_sec": 38 * 60},
    "l2_wna16": {"model": "intel", "env": {**_e13_rc1_env, "E13_L2_NO_MARLIN": "1"}, "add": ["--disable-cuda-graph"], "nan": True,
                 "chat_max_tokens": 0, "ple_modes": ["orig"], "min_room_sec": 22 * 60, "budget_sec": 25 * 60}}
E13_VLLM_CHECK = False            # arm 2 runs vLLM for real
E13_VLLM_PORT = int(os.environ.get("E13_VLLM_PORT", 8000))
CONFIG["e13"]["rc1"] = {"arms": list(E13_PROBE_ARMS), "cfg": E13_ARM_CFG}


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
# E13 rev 6b (audit B-H1): skipped; with 102 GB of pinned PLE the 178 GB cat evicts the loader's prefetched pages
if os.environ.get("Q27X_SKIP_INSTALL") != "1" and not globals().get("E13_SKIP_CAT_PREFETCH"):
    try:
        import shlex as _shlex
        _wdirs = (SGL_MODEL_PATH, SGL_DRAFT_PATH) if ENGINE_BACKEND == "sglang" else (MODEL_PATH,)   # E11-05
        _wfiles = sorted(str(p) for _d in _wdirs for p in Path(_d).glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    _tok_pkgs = [f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                 if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]]
    if ENGINE_BACKEND == "sglang" and before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        # E11-05: the kernel only tokenizes (S49 stack); SGLang runs in its own venv; vLLM is installed only for the fallback
        try:
            offline_pip_install(_tok_pkgs)
        except Exception as _e:
            print("tokenizer-only install failed -> the full S49 install:", repr(_e)[:300], flush=True)
            offline_pip_install(["vllm==0.19.0"])
    elif before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install(_tok_pkgs)
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if _p == "vllm" and ENGINE_BACKEND == "sglang":
            continue                                   # E11-05: checked by the fallback cell, only if it runs
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")

# E11-05: the SGLang server receives token ids made by this (S49) tokenizer. Check that the INT4 checkpoint has the same
# vocabulary (with added tokens) and tokenizes a rendered prompt identically; otherwise run the S49 vLLM engine.
E11_TOKENIZER_CHECK = {"ok": None, "skipped": ENGINE_BACKEND != "sglang"}
if ENGINE_BACKEND == "sglang":
    try:
        from tokenizers import Tokenizer as _HFTokenizer
        _t4 = _HFTokenizer.from_file(str(SGL_MODEL_PATH / "tokenizer.json"))
        _probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Example 1\nInput:\n0 1 2\n3 4 5\nOutput:\n5 4 3\n```python\ndef transform(grid):\n    return grid\n```"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort=REASONING_EFFORT)
        _probe += "Let me think.\n</think>\n\n```python\nimport numpy as np\n```<|im_end|>"
        # rev 2: + a real ARC prompt fragment (every train pair of the first test task, rendered as digit rows)
        _t0 = test_challenges[sorted(test_challenges)[0]]
        _probe += "".join(f"\nExample {_i + 1}\nInput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["input"])
                          + "\nOutput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["output"])
                          for _i, _p in enumerate(_t0["train"]))
        _ids_s49 = tokenizer.encode(_probe, add_special_tokens=False)
        _ids_int4 = _t4.encode(_probe, add_special_tokens=False).ids
        _same_vocab = _t4.get_vocab(with_added_tokens=True) == tokenizer.backend_tokenizer.get_vocab(with_added_tokens=True)
        # rev 2: the BPE merges too (the two files store them in different formats: "a b" strings vs [a, b] pairs)
        _norm = lambda m: [tuple(x.split(" ", 1)) if isinstance(x, str) else tuple(x) for x in m]
        _same_merges = (_norm(_load_json(SGL_MODEL_PATH / "tokenizer.json")["model"]["merges"])
                        == _norm(_load_json(MODEL_PATH / "tokenizer.json")["model"]["merges"]))
        _int4_gen = _load_json(SGL_MODEL_PATH / "generation_config.json") if (SGL_MODEL_PATH / "generation_config.json").exists() else {}
        _e4 = _int4_gen.get("eos_token_id", [])
        _same_eos = set(_e4 if isinstance(_e4, list) else [_e4]) == EOS_IDS
        E11_TOKENIZER_CHECK = {"ok": bool(_same_vocab and _same_merges and _ids_s49 == _ids_int4 and _same_eos),
                               "same_vocab": _same_vocab, "same_merges": _same_merges, "same_probe_ids": _ids_s49 == _ids_int4,
                               "probe_tokens": len(_ids_s49), "same_eos": _same_eos}
    except Exception as _e:
        E11_TOKENIZER_CHECK = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:300]}
    if not E11_TOKENIZER_CHECK["ok"]:
        print("E11 tokenizer check FAILED -> S49 vLLM FP8 engine:", E11_TOKENIZER_CHECK, flush=True)
        ENGINE_BACKEND = "vllm"
CONFIG["e11"]["tokenizer_check"] = E11_TOKENIZER_CHECK
print("E11 tokenizer check:", E11_TOKENIZER_CHECK)


In [ ]:
# ==================================================================== CELL 3 — E12: Duck package (verbatim) + ARC-2 adapter
# Tufa Labs' Duck harness as forked by Daniel Franzen (da-fr/arc-agi-3-solution @ 10882e332d, Apache-2.0,
# LICENSE + NOTICE inside). The 20 files of inference/ are byte-identical to that commit (sha256 manifest below);
# arc2_duck.py is the only new code (ARC-2 prompt, prelude with check/submit, host re-verification, driver, selection).
import base64 as _b64, io as _io, logging as _logging
E12_DIR = Path(os.environ.get("E12_DIR", "/tmp/e12_duck"))
_E12_ZIP = "..."
_E12_MANIFEST = {
"arc2_duck.py": "1c4d34796ed8df83fda50c1327f7fbccc216e6fd42d2678da2fd618577ef5e49",
"vendor/LICENSE": "3ddf9be5c28fe27dad143a5dc76eea25222ad1dd68934a047064e56ed2fa40c5",
"vendor/NOTICE.txt": "1e2b756570ed5a6d85d06e61e458021c1221a5f72ad33165ea29bb85b9053cf1",
"vendor/inference/__init__.py": "d92fed573b404870f7c6a1cef8f3e5d696195e2886e9e9183485c304fe5090f8",
"vendor/inference/agent/__init__.py": "8c561093771390b6f64fc0e87186982cd04045e8b6a2cf8d7f17e494762b4385",
"vendor/inference/agent/action_names.py": "8dd40e957046977e811eb2c2eb13fce905dbecf77f8dc7f846540f4d48af963d",
"vendor/inference/agent/noop_repeat_guard.py": "e9172d58687e9043c3445f87e11bdf74c4104d8b0bed5fb6b54745a6aa58655a",
"vendor/inference/agent/priority_pace_reference.json": "f29c16e336aef6b0dbf73ec056f554bd5192ac02f8deeda70ad152ef1214d0ab",
"vendor/inference/agent/priority_scheduler.py": "3a13659001458ea1a45ee3a812babe1b0012409eeabb23d681395796941b8e94",
"vendor/inference/agent/prompts.py": "14965a99bc55a61dad59fb79038f56c0ba1d1368350c8487b5a7b9edef0488ac",
"vendor/inference/agent/python_tool_sandbox.py": "c2208578220862aceb352138b9c3a3bb7347f1e6a4ad09d1d9cb3f2ee3507477",
"vendor/inference/agent/runtime_state.py": "b5c55d561a63e2dcf0422ee7fd339923a9fb909c779302bbf02b8ea49ccc607f",
"vendor/inference/agent/tool_agent.py": "485ef8a98308654b9dbb69535bc3c6ed3723c80bbcd5a4538e3ce181e6fda615",
"vendor/inference/agent/vision_context.py": "51d54a7aa9f1cd8e41b3fcc00ffd1a9cd1ada6fcde261d09faa49561fd843466",
"vendor/inference/utils/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
"vendor/inference/utils/animation.py": "6c14b613b3a7c66ce3373162d6faa1469a1ddb3ca4ab882299998c5cf14add96",
"vendor/inference/utils/frame_diff.py": "6ac3112773ffa6b1562b1e8f35a8018bd0dec9de7a78b12e7b13c529d4f46af8",
"vendor/inference/utils/grid_utils.py": "1e43809c9a4a463c65f897c48abb8e6cf4b28a32ac4ec32a07e9adcf06ab0c03",
"vendor/inference/utils/openai_compat.py": "fea0a75081372769fd854997b99e7826b334dcc5966876f31f587d19ae961165",
"vendor/inference/utils/retained_functions.py": "0d8b267cb387565c04d4977116020cc04d84ad9d9175d2bf4bc4613703cc5a01",
"vendor/inference/utils/run_artifacts.py": "789d541b395043379c23f5b0b24044e50fadab02e7f86f64cdd54714f4cab339",
"vendor/inference/utils/segmentation.py": "0d19669e5a36351e41a6b94b7c68f5b258829de78441a03a8749687c262b452c",
"vendor/inference/utils/viewer_artifacts.py": "cb5043dab8f5294608cda3675408ced904b849ae4985829b4ffd2d84b8e5d807"
}
if E12_DIR.exists():
    shutil.rmtree(E12_DIR)
with zipfile.ZipFile(_io.BytesIO(_b64.b64decode(_E12_ZIP))) as _z:
    _z.extractall(E12_DIR)
_bad = [p for p, h in _E12_MANIFEST.items() if hashlib.sha256((E12_DIR / p).read_bytes()).hexdigest() != h]
assert not _bad, ("E12 package hash mismatch", _bad)
sys.path[:0] = [str(E12_DIR), str(E12_DIR / "vendor")]
import arc2_duck as E12A
# Duck reads its settings at import: set them first. The base URL is the engine cell's (same port); the stream count is
# set again from the engine's concurrency before the gate is first used (read at call time).
os.environ.update(E12A.duck_env(f"http://127.0.0.1:{int(os.environ.get('E11_PORT', 30000))}/v1", "pending", SGL_MAX_RUNNING))
_E12_TA, _E12_DP, _e12_run_sandboxed, _ = E12A._import_duck()
E12_SANDBOX_LIMIT = E12A.install_sandbox_memory_limit(E12_SANDBOX_MEM_MB)   # rev 2: every sandbox child, before its code


class _E12LevelCountFilter(_logging.Filter):   # Duck's per-game level-count warning has no ARC-2 meaning (priority uses 10)
    def filter(self, record):
        return "total level count unavailable" not in record.getMessage()


_logging.getLogger(_E12_TA.__name__).addFilter(_E12LevelCountFilter())

# sandbox self-test with our own scripted code (the engine cell runs only if it passes)
_t = time.perf_counter()
try:
    _st_task = test_challenges[sorted(test_challenges)[0]]
    _st = _e12_run_sandboxed(
        code=E12A.task_prelude(_st_task) + "def _e12_ident(g):\n    return g\nres = check(_e12_ident, verbose=False)\nprint(len(res), sum(1 for r in res if 'ok' in r))",
        timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [], "last_action_call_result": {}},
        action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[], retain_imports=True)
    _n = len(_st_task["train"])
    E10_SANDBOX_SELFTEST = {"ok": (_st.get("stdout") or "").strip() == f"{_n} {_n}" and not _st.get("error"),
                            "stdout": (_st.get("stdout") or "")[:200], "error": (_st.get("error") or "")[:500],
                            "sec": round(time.perf_counter() - _t, 2), "python": sys.version.split()[0]}
except Exception as _e:
    E10_SANDBOX_SELFTEST = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:500]}
# rev 2: memory limit evidence (reported; the run continues without it rather than not at all). The 5 GiB request is
# made only after the read-back confirmed the limit, so it can never really allocate.
_mem = {"limit": dict(E12_SANDBOX_LIMIT)}
try:
    _st_kw = dict(timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [],
                                                     "last_action_call_result": {}},
                  action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[],
                  retain_imports=True)
    if E12_SANDBOX_LIMIT.get("readback_ok"):
        _big = _e12_run_sandboxed(code="x = bytearray(5 * 1024 ** 3)\nprint(len(x))", **_st_kw)
        _small = _e12_run_sandboxed(code="x = bytearray(128 * 1024 ** 2)\nprint(len(x))", **_st_kw)
        _mem.update(over_limit_rejected="MemoryError" in (_big.get("error") or ""),
                    small_ok=(_small.get("stdout") or "").strip() == str(128 * 1024 ** 2))
        _mem["ok"] = bool(_mem["over_limit_rejected"] and _mem["small_ok"])
    else:
        _mem["ok"] = False
except Exception as _e:
    _mem.update(ok=False, error=f"{type(_e).__name__}: {_e}"[:300])
_mem["limit"] = dict(E12_SANDBOX_LIMIT)
E10_SANDBOX_SELFTEST["memory_limit"] = _mem
if E12_SANDBOX_MEM_MB > 0 and not _mem.get("ok"):
    print("E12 WARNING: sandbox memory limit NOT verified:", json.dumps(_mem), flush=True)
    if os.name == "posix":   # rev 3 (GPT re-audit): on Kaggle (Linux) never run model code without the limit
        E10_SANDBOX_SELFTEST["ok"] = False
        E10_SANDBOX_SELFTEST["error"] = "memory limit not verified (fail closed): " + json.dumps(_mem)[:300]
        print("E12: solver disabled (fail closed); CELL 0 fallback submission stays", flush=True)
CONFIG["e12"]["sandbox_selftest"] = E10_SANDBOX_SELFTEST
print("E12 package OK:", len(_E12_MANIFEST), "files | sandbox self-test:", json.dumps(E10_SANDBOX_SELFTEST), flush=True)


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# ==================================================================== CELL 5 — E13 engine: SGLang 0.5.21 + Qwen3.8-Flash-Next (Franzen's launch)
# Duck-only (no Poetiq shim). Reused verbatim from E12 (Kaggle-proven): E11Fail .. e11_install, E11Server, e11_kill_server,
# e11_decode_summary. New: FA2 + patch #3, the top-k test, a host-RAM guard, Franzen's launch with the L4 deltas, a smoke.
import queue as _queue, signal as _signal, statistics as _statistics, types as _types, urllib.request as _urlreq, base64 as _b64
E11_HARNESS = os.environ.get("E11_HARNESS") == "1"            # local CPU harness only: fake server, no install, no GPU
SGL_PORT = int(os.environ.get("E11_PORT", 30000))
SGL_URL = f"http://127.0.0.1:{SGL_PORT}"
E11_SERVER = None
ENGINE_READY = False; llm = None; engine = None; CONCURRENCY = 0
ENGINE_INFO = {"backend": None, "e13": {"stage": "not started"}, "kv_tokens": None, "concurrency": None, "fatal": None,
               "load_sec": None, "tokenizer_check": E11_TOKENIZER_CHECK}

class E11Fail(Exception):
    pass


def _e11_post(path, payload, timeout=30):
    req = _urlreq.Request(SGL_URL + path, data=json.dumps(payload).encode(), method="POST",
                          headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=timeout) as r:
        return r.read()


def _e11_gpu_used():
    try:
        return [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                               capture_output=True, text=True, timeout=30).stdout.split()]
    except Exception:
        return []


def _e11_gpu_free(max_sec=180):
    t = time.perf_counter()
    while time.perf_counter() - t < max_sec:
        u = _e11_gpu_used()
        if E11_HARNESS or (u and max(u) < 1024):
            return {"free": True, "sec": round(time.perf_counter() - t), "used_mb": u}
        time.sleep(5)
    for _pat in ("sglang.launch_server", "sglang::"):
        subprocess.run(["pkill", "-9", "-f", _pat], capture_output=True)
    time.sleep(15)
    return {"free": False, "sec": round(time.perf_counter() - t), "used_mb": _e11_gpu_used(), "pkill": True}


# ---- install (= E11-03/04 C1): a venv from the wheelhouse lock, --no-deps; CUDA 13 pip runtime as CUDA_HOME; link shims
def e11_install():
    if E11_HARNESS:
        return Path(sys.executable), dict(os.environ), {"harness": True}
    shims, t0 = {}, time.perf_counter()
    venv = Path("/tmp/sglang-venv"); vpy = venv / "bin" / "python"
    # Kaggle drops '+' from dataset file names (flashinfer_jit_cache-0.6.18+cu130 -> ...0.6.18cu130); rebuild the original
    # names from SHA256SUMS as links in a find-links folder, checking the hash of every renamed wheel
    fl = Path("/tmp/e11-wheels"); fl.mkdir(parents=True, exist_ok=True)
    renamed = []
    for line in (SGL_WHEELHOUSE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, name = line.split("  ", 1); name = name.strip()
        if not name.endswith(".whl"):
            continue
        s = SGL_WHEELHOUSE / name if (SGL_WHEELHOUSE / name).exists() else SGL_WHEELHOUSE / name.replace("+", "")
        if not s.exists():
            raise E11Fail(f"wheel missing from the dataset: {name}")
        if s.name != name:
            d = hashlib.sha256()
            with open(s, "rb") as f:
                for b in iter(lambda: f.read(16 * 1024 * 1024), b""):
                    d.update(b)
            if d.hexdigest() != h:
                raise E11Fail(f"sha256 mismatch for {s.name}")
            renamed.append(f"{s.name} -> {name}")
        if not (fl / name).exists():
            (fl / name).symlink_to(s)
    shims["renamed_wheels"] = renamed
    if not vpy.exists():
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
        r = subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(fl),
                            "--requirement", str(SGL_WHEELHOUSE / "requirements.lock"), "--no-deps", "--only-binary", ":all:",
                            "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts", "-q"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise E11Fail("offline SGLang install failed: " + (r.stderr or r.stdout)[-1500:])
    site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True).stdout.strip())
    cuda_home = site / "nvidia" / "cu13"
    # E11-02 BUILD_PROBE: the pip CUDA runtime ships lib/libcudart.so.13 only; JIT links need libcudart.so and lib64
    if not (cuda_home / "lib" / "libcudart.so").exists():
        (cuda_home / "lib" / "libcudart.so").symlink_to("libcudart.so.13"); shims["libcudart.so"] = "created"
    if not (cuda_home / "lib64").exists():
        (cuda_home / "lib64").symlink_to("lib"); shims["lib64"] = "created"
    # E10 run 1: FlashInfer's JIT link needs -lcuda; Kaggle has only libcuda.so.1
    libcuda = Path("/tmp/e11-libcuda")
    have = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs", "/usr/lib/x86_64-linux-gnu", "/usr/lib64")
            if Path(d, "libcuda.so").exists()]
    if have:
        shims["libcuda.so"] = f"exists in {have[0]}"
    else:
        real = None
        try:
            for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                if "libcuda.so.1" in line and "=>" in line:
                    real = line.split("=>")[-1].strip(); break
        except Exception:
            pass
        for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            if real is None and Path(d, "libcuda.so.1").exists():
                real = str(Path(d, "libcuda.so.1"))
        if real:
            libcuda.mkdir(parents=True, exist_ok=True)
            if (libcuda / "libcuda.so").is_symlink() or (libcuda / "libcuda.so").exists():
                (libcuda / "libcuda.so").unlink()
            (libcuda / "libcuda.so").symlink_to(Path(real).resolve()); shims["libcuda.so"] = f"shim -> {real}"
        else:
            shims["libcuda.so"] = "libcuda.so.1 NOT FOUND"
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH")}
    env.update({
        "PATH": os.pathsep.join([str(venv / "bin"), str(cuda_home / "bin"), os.environ.get("PATH", "")]),
        "CUDA_HOME": str(cuda_home), "CUDA_PATH": str(cuda_home),
        "LIBRARY_PATH": os.pathsep.join(x for x in (str(libcuda), str(cuda_home / "lib"), os.environ.get("LIBRARY_PATH", "")) if x),
        "LD_LIBRARY_PATH": os.pathsep.join(x for x in (str(cuda_home / "lib"), os.environ.get("LD_LIBRARY_PATH", "")) if x),
        "TORCH_CUDA_ARCH_LIST": "8.9", "MAX_JOBS": "4",
        "SGLANG_CACHE_DIR": "/tmp/e11-cache/sglang", "CUDA_CACHE_PATH": "/tmp/e11-cache/cuda",
        "TRITON_CACHE_DIR": "/tmp/e11-cache/triton", "TORCH_EXTENSIONS_DIR": "/tmp/e11-cache/torch_ext",
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
        "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "PYTHONUNBUFFERED": "1",
    })
    shims.update(cuda_home=str(cuda_home), venv_python=str(vpy), sec=round(time.perf_counter() - t0, 1))
    return vpy, env, shims



class E11Server:
    def __init__(self, cmd, env, tag):
        self.cmd, self.env, self.tag, self.proc, self.log_f = cmd, env, tag, None, None
        self.log_path = OUT / f"sglang_server_{tag}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     cwd=None if E11_HARNESS else "/tmp", start_new_session=True)

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout):
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"server exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
                return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                time.sleep(2 if E11_HARNESS else 5)
        return False, "timeout"

    def kill(self, wait=True):
        if self.alive():
            for sig, w in ((_signal.SIGTERM, 30 if wait else 0), (getattr(_signal, "SIGKILL", _signal.SIGTERM), 30)):
                try:
                    if hasattr(os, "killpg"):
                        os.killpg(self.proc.pid, sig)          # the whole group: TP workers, scheduler, detokenizer
                    else:
                        self.proc.kill()
                except Exception:
                    pass
                try:
                    self.proc.wait(timeout=w if w else 0.1)
                    break
                except Exception:
                    continue
        if self.log_f:
            try:
                self.log_f.close()
            except Exception:
                pass
            self.log_f = None


def e11_kill_server(wait=True):
    if E11_SERVER is not None:
        E11_SERVER.kill(wait=wait)



_E11_KV_RE = re.compile(r"([A-Za-z#][A-Za-z#_ ()/\-]*?):\s*(-?\d+(?:\.\d+)?)")


def e11_decode_summary():
    """Decode rate by running level and by mean context per running request, from the server's TP0 decode lines."""
    if E11_SERVER is None:
        return None
    rows = []
    for line in E11_SERVER.log_text().splitlines():
        if "Decode batch" in line and re.search(r"TP0[ \]]", line):
            rows.append({k.strip(): float(v) for k, v in _E11_KV_RE.findall(line.split("Decode batch", 1)[1])})
    def med(v):
        return round(_statistics.median(v), 1) if v else None
    bins = {}
    for r in rows:
        n, ft, g = r.get("#running-req"), r.get("#full token"), r.get("gen throughput (token/s)")
        if not n or ft is None or g is None:
            continue
        ctx = ft / n
        k = "<10k" if ctx < 10000 else "10-20k" if ctx < 20000 else "20-40k" if ctx < 40000 else "40-60k" if ctx < 60000 else ">=60k"
        bins.setdefault(f"running={int(n)}" if n < SGL_MAX_RUNNING else "running=max", {}).setdefault(k, []).append(g)
    out = {"n_lines": len(rows),
           "running_hist": {str(int(k)): sum(1 for r in rows if r.get("#running-req") == k) for k in sorted({r.get("#running-req", 0) for r in rows})},
           "gen_tok_s_median_at_max": med([r["gen throughput (token/s)"] for r in rows
                                           if r.get("#running-req") == SGL_MAX_RUNNING and "gen throughput (token/s)" in r]),
           "accept_len_median": med([r["accept len"] for r in rows if "accept len" in r]),
           "queue_max": max((r.get("#queue-req", 0) for r in rows), default=None),
           "retract_lines": sum(1 for l in E11_SERVER.log_text().splitlines() if "retract" in l.lower()),
           "by_context": {lvl: {k: {"lines": len(v), "gen_tok_s_median": med(v)} for k, v in d.items()} for lvl, d in bins.items()}}
    try:
        write_json_atomic(OUT / "sglang_decode_summary.json", out)
    except Exception:
        pass
    return out




def metrics_snapshot():                                  # SGLang /metrics (CELL 6's heartbeat matches name substrings)
    if not ENGINE_READY:
        return []
    try:
        txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=10).read().decode("utf-8", "replace")
        rows = []
        for line in txt.splitlines():
            m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)", line) if line and not line.startswith("#") else None
            if m and not m.group(1).endswith(("_bucket", "_created")) and any(
                    k in m.group(1) for k in ("num_running_reqs", "num_queue_reqs", "gen_throughput", "token_usage", "cache_hit_rate")):
                try:
                    rows.append({"name": m.group(1), "labels": m.group(2), "value": float(m.group(3))})
                except ValueError:
                    pass
        return rows
    except Exception as e:
        return [{"error": repr(e)[:300]}]


def _e13_sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(16 << 20), b""):
            h.update(b)
    return h.hexdigest()


def e13_extras(vpy):
    """FA2 (QSA decode on sm89, upstream PR #36968) + FA4 restore + Franzen patch #3 with his SHAs."""
    if E11_HARNESS:
        return {"harness": True}
    names = (E13_FA2_WHEEL, E13_FA2_WHEEL.replace("+", ""))           # Kaggle drops '+' from file names
    cands = [d / n for d in E13_FA2_DIRS for n in names]
    if Path("/kaggle/input").is_dir():                                 # rev 6: any attached copy (found by name, checked by sha)
        cands += [f for n in names for f in sorted(Path("/kaggle/input").rglob(n))]
    w = next((c for c in cands if c.is_file() and _e13_sha(c) == E13_FA2_SHA), None)
    if w is None:
        raise E11Fail("FA2 wheel (sha256 74984f62...) not found under " + ", ".join(str(d) for d in E13_FA2_DIRS)
                      + " or /kaggle/input: attach foysalemonshanto/flash-attn-2-8-3-post1-cu13-torch2-13-cp312")
    fl = Path("/tmp/e13-fa2"); fl.mkdir(exist_ok=True)
    if not (fl / E13_FA2_WHEEL).exists():
        (fl / E13_FA2_WHEEL).symlink_to(w)
    pip = [sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--no-deps", "-q", "--disable-pip-version-check"]
    r = subprocess.run(pip + ["--find-links", "/tmp/e11-wheels", str(fl / E13_FA2_WHEEL)], capture_output=True, text=True)
    if r.returncode:
        raise E11Fail("FA2 install failed: " + (r.stderr or r.stdout)[-800:])
    fa4 = sorted(Path("/tmp/e11-wheels").glob("flash_attn_4-*.whl"))   # FA2 ships 21 old flash_attn/cute files: restore FA4's
    if fa4:
        r = subprocess.run(pip + ["--force-reinstall", str(fa4[0])], capture_output=True, text=True)
        if r.returncode:
            raise E11Fail("FA4 restore failed: " + (r.stderr or r.stdout)[-800:])
    site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True).stdout.strip())
    g = site / "sglang/srt/layers/quantization/gptq/schemes/gptq_moe.py"
    h = _e13_sha(g)
    if h == E13_PATCH3[0]:
        t = g.read_bytes(); assert t.count(b"dtype=torch.half") == 2
        tmp = g.with_name(g.name + ".e13tmp"); tmp.write_bytes(t.replace(b"dtype=torch.half", b"dtype=params_dtype")); os.replace(tmp, g)
        h = _e13_sha(g)
    if h != E13_PATCH3[1]:
        raise E11Fail(f"patch #3 after-sha mismatch {h[:12]}")
    probe = ("import json, torch, flash_attn\nfrom sglang.srt.layers.attention.qwen_sparse_attn_backend import "
             "_resolve_flash_attn_varlen_func as r\nprint(json.dumps({'fa2': flash_attn.__version__, 'resolves_to': r().__module__,"
             "'cxx11abi': torch.compiled_with_cxx11_abi()}))")
    r = subprocess.run([str(vpy), "-c", probe], capture_output=True, text=True, env=_E13_ENV, timeout=900)
    info = json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else {"probe_error": (r.stderr or "")[-800:]}
    if not str(info.get("resolves_to", "")).startswith("flash_attn.") or "cute" in str(info.get("resolves_to")):
        raise E11Fail("QSA decode does not resolve to classic FA2: " + json.dumps(info)[:600])
    return info


E13_TOPK_SRC = __import__("zlib").decompress(_b64.b64decode("eNqtWtuS2zYSfddXIPTDko5Ek5qJM5ZXqUptHK/XqWw2Hu+LSkVDIigxokiaAGc0mdK/72mAN0nUWN6KHzwkCHQ3+nrQkGVZb/wrto2XRTbKCxEl8Wqt2K3/4nY8Yf/58CNbJNlyw1SWjzbM3ky/88cOW2ZFIZYqFVKyLGVqHUv29rePQ7YQUVYIxtMHts1CkTB8SDIeitAdDG59NmUf3v7C0xXz3O/csc/+9e6WfYq4VAEYbD4xW60F24gixVJiXkohGUiyUCxBD4RDRlLGScLu1yI10unFbDplEO41WMpSsGdXL2+871kh8qxQciDjRKSKhUWWa5GXWbrEQMGVCJnEfoQeltubV47LbseQ9OeEy/W7NBIF+wQGwSYo+GolwgCrUgmhtnYolCi2cRpLFS+nt0UphkzFIlgUgm+m3nAQSh6sCp6vA8kjMf2ZJ1I4n4ZsK7gsC7DO0uSB2ZzJ+1gt1ywVIpSMlICdF3+TjOd5kd3xxIH+/l3wZSKgo1iM+D0vhDNhOaQrsvuhXiNFAqvQhoRi21IqtuZ3AsZN7c2QQQEOoz3HQpKlWJnGn0uj0zhlM89MGZoBJRnYlmKwLRMVNwTF55InmlkznkX6HSo64KRXS5e92fGlAoNQ7MzqWD2QW6SZgnU+lzGpgfYkscePkq8EdvWg1rDG6B3b5tq4bv7ARqOsVOz3Nx8+/nLr/iHp+yiKd6ok2/307nc2G41gbsEh+tib0+syL0dQSqSEVPOBZVmDeEv+wHixynkhRf1O5OrnTNZP8qF5VPG2mQz7L8WCLzeDZiQrluvB4D3TLjj45c2vb2//+QFvsysPah171zdDduO/Gg+Z//Lq5hpD19/Rlyvv1RU+XV3f+C/njLFncMwtHFxKKEX7tnxtvoKYf/VqOH55DW4bkUr2gl3jmzcEBXw0/j746d2HW83YWvFSypin1pBZsDT5Kz12HZ/eC5GUgdTKaF/L7TW9pIIXASxDzzxJAm0+az4YDEIRsS3fiMCEjh0iAobkiFJbf6XWQ7YS6RBxexcv4agDbI7BAFF+NWaz7sS5yzTT5/AepCHtSxT62mPg3ZrDhEEmlt3hfS04QgROR6vsz+7GgSbk50LZocPs7WcOX3FcsjWxjCNGslFyaDVipKF/O6hKG89FTIepfbIDAUVlxRRPjl4kki7FWq/nCH4tvQPjTAzFZ8ik+LzlyLXaAAhQFol7lsCJoRjSV5XsIh4niAYm1zwXPSL9KYpMHsjkNLPWCMeLRWd/RwL3/JbFDMvn5P2ux77tUolTZeObC/PZjtOrAiPjt/BvMbpiz5ELvkYIb+y4EUqMsvsU2vXuSp9QKJxGKxWpNxUrrmIkydEPjC8owax5YhKaCT/jf0xQFsN8r8/S5Ijnvej6RPAR813P0ToZ+eS+/viGPX9ORfHsHigkJ38B84vZNsHfyzUrYZ1LIwdGvSZ+1z1s2rzSxycDxOhxWIEy2k4vOOoQ+y/VmzdFkRU6G5mJhUB5SNnOVZltUtHU/EFmUg+5mBo22oGu4EkIQBWvyqyUcCeT5pZrAYRBMtgm2dWi4AE5CrRkm95+N/w4S2h/SFKozyOsBWjJFonYSmbPKE6wa50tayTVJqwFDyl9z/WLDFDBwM/8iekPPhoxaq93MWg7jVD1ayWaedXECEYtqNLDYCtha9quThUzb+606kzBgsJWc50t5s5rds9Tyg5U3t8PWSduaWtTIxhmNsOouRjGR8r07Icp89pvpIl2h1WOxgI3LbcisR32zVTzawWqV7kAQwJuFyFPlhDosbNqT8sead3ecs5RJmCjMxIN0mYcnUQYFNOM8h2NQuTUOS+BZeAMpQsqRKTPQ67GrQzAIrpOd3cdkZ5gEZZ5Ei9RCUwlPKRP4IlmH65fdXK4BC4xNp4thpplkqUr25k7s44xTIxEzSrCWu2qSTofals4boXmNNUTCpVAhoQOZhuiDIny0Q5PTakJt2AyjCMAboQJzKP382jcHI/aS92Xqz27k1ro6hMem09OR03PmJ3zMIxx2sgzGasYiLHCnTqmRThhzdEDUBvWzMmgRQiUgYllKnOxjKNYhK91QTBep9fi+Y7E3OKwQQxkkil5YKFT6yCy620/WogLFCTo2ELc4glQyapzBF7pcX+QwrC6SkhFmQbAnYmd8i3yWNSgLFK4hr8YrHCxwWQBZaOpjVrge05lEQLNiBiLKBF/Tctachy48DqD4S2CE0E9ArBqqUzxzsheE0KCb9znbZ32kZa2PIUjBBInGnvsjV/6nv+qTUQ6+VEuqsVrdUWfTTKjCRWWPlSlPhFShsUEjXhPvaw6002/CqaeUHnGzJnPwCwcCJVOefAOuUYowFFwdsNhL4F96Py2jXeCknqqxE7VKfmUbEL4vVZaVCaJbYqc04rVLU9ITihOJ1TgZVqNPzD/dP81m9nIn2st7Gy/ps5evGBXZ6WiP229PJ1GbiEppZCjiaLAEk+/Utn5FRX7ZIXOSjhAuaiGUaDDSBR2D2VyjLZCVd7s9G9OFQ/9H7RqQsIQUdot2T126GjS7Nag8DLk1nnamrm2C0105UO6XBdZGv8p7H4OYrcUucJBmP5QFuJA8ufpG5VG1iN5gC0cNwgoOoNgP2GPYm/NJjhUzs8u172H3q8GWvTjGQIz4a5ffrK0i0xqK5QklDNyfCD7cIe0OzN/TI2H11Aoo4A6/ZSgZwhxfuvatfRR4ClTmVmwlK8rep3rnrZYJl3KBGFcSLvNjmIHeYNso3s3zgUml2ioIH0bzSEJVlisAl1WFfH4oKOoHq7AGIahrna0k3qfZN39Z1EOwyqTyqiOEFmT0ir2hrvOIhalX7zXCVif8CmqaM3XMCVfxBqrQsqWYR1IxdFdow/6bEnDut2FEd3rwjsMTpWOh7PJ9Xx/MU/YK+dq7f6RAae1BkNYkL72wSMpAH9o5/vdo9nw3s2V1eN7Uizh+XZPDsJZTNHRpMqPdco5oUBFT9fLE4V3tX1oG10/K4rWpM6bVjtSl+vBk8aO06WiePUCCkTDzqZH8qGtDLAh1OQk0VLRscz3PA9HLmwanSWaJOhUhM/4ux+cDSgkX0zojyLoftbFA/PzQUoTZhbOy/BuYVHtsc+mnOME7dRQknIgHfJ0PXFItN7CUsl/xPJ8IjiV7WhkNhmfZla9+S72ObN7Pa+aUSM9ej3AcZhU4TjTF4YPJPzhBMu1R8p/8Jx8n1FrSLeoFRoWS7YoNVAeIp9TsTT9X4N5UEBBs8KDID6sMCtazGs9L9ON5IMemal+35xWv0rsY6AoNzE2SJFtpRnTa4xr0ZnoFFcaQWocad6Cql1Vj9ZeSpY2tA5K/CU48/uDk2kD8qh36lVd1sGTAPGoHXoBUvx/oFxNZXpMjABQ0AKgqyPocxbPfBGOrBoR9aR/fPzpx7fkfp0pxjvaOdo97dWRCF8AVl1gbh+0oL/Qd266TUf86vqa5Q+B/TVQ/jCFr9wqztDMuAi7wY1njdf2xHscPQ2kTnNQh2Lr+R3KX0CJerkJkfkX4OF1DQ+r2MXSpmVPTQ9DlOfUh6ruQdwfixU6Eqn6jd5qbM5zgn0Br77Zlr6Cobio7m06sKlnbl21LTJKxHHEn1rW2dl1WcRVFoULAqVdNvbOLute8WAturTQ3dSSsLLATV1J3rUWST610M7lyaTKhW0a7Dq+vkckTMkBo+mWhek0VjEnfeWu1heJIO06ses7IEp6xveqHEYHZr/zPG6fa4ykj9naqObQva/zMSeQGNT7ar0ADBjl6Ly02tjOMsL1Cd8uQlwkwgsnNVhVuKSyZ5XDo1Uy63Z5dIdHN/qoyQdJqvZeM4Recd1FpDHncrx4iptNZnwPuHXChVLlyO878c6dyzkedzfbziaodPQXVZa1SVHOpHvUN5fDzUF/7FUXqnAoc+GEk7+n3aPpzdGlcdWgY/b9mqv6KibJMtwdJPHm6FRB2ZMMZti7SzTlBCWlHV2/jsbehIJ75NMAVOLRm37y56+bZnanNpPnzWq3m1eu0cbDyUw4JM1qWkiWVrfugAUkFoKF/ugsCqaIJ/dMK6mTYBvi4yPiRtP6BYTN2+WkD5v8tecT0nhKAd0KJlRgvugMaXsO7qYsQL4c/38LIFXY/bMxgy9iuqjGmk6VPe44REW2ZXKVwONc86MF6cKFXK4UshPcxm36i6y6KW4GDggp/ziAm3naUYZ0u76ZvteKqg5dUw2Njzq556zsBw1BSrB+ZQXuNh3D1hr9NE3KanDtAXg9Q7+juuPiNunlUWXChol160+0qZobd5euW7kKQA9t6JGPS/TJ/AkL1Vqnn3LE9FOOQ/vFFI0CgBqu0M7Rnc6+33tYpr/lHCMB0EFHkD5Neg4FB8Hx2EXOLUvcOlJnmp3he3hqOwyMbmpT4yqt9ebNLIqOroLbLPkUSpXuuVZgB2REMbEGC+2u72nx07+NYb2/jfmS/jo+PQ5a/VUqI98bX+7bl/v3k7z+Aj8fX+7nVOg24sEgbES6gRYdk1P30HCinGZj7sHFRIwLC2QQoHEbno7yrszF2MEJVxfSI+FnFm6Ot7x4qBz54HqgOGwPnFwWFEcn6KexhLV4CKpeymOof/thL2eH/RS0HM3N3SLLEvpaYWNHK2ipN9CcxjVcn5n2zJw6h+HlyEIfaZqLhr0JRfq9kBuW29w2mobnw5I2d4GPsfd7y9zAogpMfcMqLwjxNOuk/biZMPtOm6hRrHNkoLvaQLoXcmf2hh9Y3entGRvHCldGtl65OXaKvdNlXvn48NjnWww6ZDOBFgjOD5qTaNm0TktwagBm9bFDt8yDgI4WQVA1DvDDKRetVWWbA4cz+B+7asva")).decode()

def e13_topk_test(vpy):
    """User-mandated: SGLang's JIT fast_topk<512> (the QSA block top-k) vs a tie-aware torch oracle (#36807). Recorded only."""
    if not E13_TOPK_TEST or E11_HARNESS:
        return {"skipped": True}
    p = OUT / "mp_topk.py"; p.write_text(E13_TOPK_SRC, encoding="utf-8")
    try:
        r = subprocess.run([str(vpy), "-I", str(p), "--out", str(OUT / "e13_topk.json"), "--fixtures", str(OUT / "topk_fixtures")],
                           capture_output=True, text=True, env=_E13_ENV, timeout=1500)
        (OUT / "e13_topk.log").write_text(r.stdout + "\n--- stderr ---\n" + r.stderr, encoding="utf-8")
        rep = json.loads((OUT / "e13_topk.json").read_text())
        return {k: (rep.get(k) or {}).get("summary") if isinstance(rep.get(k), dict) else rep.get(k) for k in ("t1", "t2")} | {
            "graph": rep.get("graph"), "errors": [e[:300] for e in rep.get("errors", [])]}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"[:600]}


def e13_host_ram_guard():
    ple = sum(f.stat().st_size for f in SGL_MODEL_PATH.glob("model-ple*.safetensors"))   # BF16 model-ple-*, FP8 model-plefp8-*
    avail = None
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemAvailable"):
                avail = int(line.split()[1]) * 1024
    except Exception:
        pass
    cg = None                                          # rev 6b (B-L2): the cgroup limit can be below MemAvailable
    try:
        mx = open("/sys/fs/cgroup/memory.max").read().strip()
        if mx.isdigit():
            cg = int(mx) - int(open("/sys/fs/cgroup/memory.current").read().strip())
    except Exception:
        pass
    eff = min(x for x in (avail, cg) if x is not None) if (avail is not None or cg is not None) else None
    need = ple + 20 * 10 ** 9
    return {"ple_gb": round(ple / 1e9, 1), "mem_available_gb": round(avail / 1e9, 1) if avail else None,
            "cgroup_free_gb": round(cg / 1e9, 1) if cg is not None else None,
            "need_gb": round(need / 1e9, 1), "ok": E11_HARNESS or (eff is not None and eff >= need)}


def e13_flags():
    """Franzen's launch (his notebook cell 12) with the L4 deltas only (PLAN rev 4.1 sections 2 and 3b)."""
    run = SGL_SERVER_MAX_RUNNING
    gb = [n for n in sorted({1, 2, 4, 7, 8, 9, 10, run}) if n <= run]
    f = ["--model-path", str(SGL_MODEL_PATH), "--load-format", "safetensors",
         "--model-loader-extra-config", json.dumps({"enable_multithread_load": False}),
         "--served-model-name", "flashnext", "--host", "127.0.0.1", "--port", str(SGL_PORT),
         "--tensor-parallel-size", "4", "--ep-size", "4",                                               # D4
         "--dtype", "bfloat16", "--quantization", "auto-round", "--kv-cache-dtype", "fp8_e4m3",
         "--mem-fraction-static", E13_MEM_FRACTION, "--context-length", str(SGL_CONTEXT_LEN), "--page-size", "64",
         "--max-running-requests", str(run), "--chunked-prefill-size", str(E13_CHUNK), "--max-prefill-tokens", "16384",
         "--cuda-graph-max-bs-decode", str(run), "--cuda-graph-bs-decode", *map(str, gb),
         "--mamba-ssm-dtype", "bfloat16", "--max-mamba-cache-size", str(run * 5),                     # extra_buffer: 5 slots
         "--mamba-radix-cache-strategy", "extra_buffer", "--mamba-track-interval", "64",
         "--mamba-backend", "triton", "--linear-attn-decode-backend", "triton", "--linear-attn-prefill-backend", "triton",   # D5
         "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
         "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
         "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
         "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}),
         "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
         "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging",
         "--weight-loader-prefetch-checkpoints", "--weight-loader-prefetch-num-threads", "1",   # rev 6b (B-H1): 4 shards in flight
         "--weight-loader-drop-cache-after-load"]
    if (SGL_MODEL_PATH / "chat_template.jinja").exists():
        f += ["--chat-template", str(SGL_MODEL_PATH / "chat_template.jinja")]
    return f


def _e13_caps(log):
    tp0 = "\n".join(l for l in log.splitlines() if "TP0" in l) or log
    kv = re.findall(r"KV Cache is allocated\. dtype: ([\w.]+), #tokens: (\d+), K size: ([\d.]+) GB, V size: ([\d.]+) GB", tp0)
    lw = re.findall(r"Load weight end.*?avail mem=([\d.]+) GB, mem usage=([\d.]+) GB", tp0)
    cap = re.findall(r"max_running_requests is capped to (\d+)", log)
    gr = re.findall(r"Capture [\w ]*?(?:cuda|CUDA) graph end\..*?mem usage=([\d.]+) GB, avail mem=([\d.]+) GB", tp0)
    mb = re.findall(r"Mamba Cache is allocated\. max_mamba_cache_size: (\d+), conv_state size: ([\d.]+)GB, ssm_state size: ([\d.]+)GB", tp0)
    pe = re.findall(r"Memory pool end\. avail mem=([\d.]+) GB", tp0)
    return {"mamba_slots": int(mb[0][0]) if mb else None, "mamba_gb": round(float(mb[0][1]) + float(mb[0][2]), 2) if mb else None,
            "kv_gb": round(float(kv[0][2]) + float(kv[0][3]), 2) if kv else None, "pool_end_avail_gb": float(pe[0]) if pe else None,
            "kv_dtype": kv[0][0] if kv else None, "kv_tokens": int(kv[0][1]) if kv else None,
            "post_weight_avail_gb": float(lw[0][0]) if lw else None, "weight_gb": float(lw[0][1]) if lw else None,
            "capped_running": min(int(x) for x in cap) if cap else None,
            "post_graph_avail_gb": float(gr[-1][1]) if gr else None, "oom": "out of memory" in log.lower()}


def e13_smoke():
    """'!!!!' (the E3 symptom) or a wrong continuation = not ready: raw greedy counting + one chat answer."""
    def post(path, body, timeout):
        req = _urlreq.Request(SGL_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
        with _urlreq.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read())
    g = post("/generate", {"text": "1, 2, 3, 4, 5, 6, 7, 8,", "sampling_params": {"max_new_tokens": 32, "temperature": 0}}, 600)
    text = g.get("text") or ""
    count_ok = text.strip().startswith("9, 10, 11")
    c = post("/v1/chat/completions", {"model": "flashnext", "max_tokens": 4096,
                                      "messages": [{"role": "user", "content": "What is 2 + 2? Answer with the number only."}]}, 1200)
    m = (c.get("choices") or [{}])[0].get("message") or {}
    ans = (m.get("content") or "") + " " + (m.get("reasoning_content") or "")
    garbage = lambda t: bool(t.strip()) and max(t.count(ch) for ch in set(t.strip())) / len(t.strip()) > 0.5 and len(t.strip()) > 8
    out = {"count_text": text[:80], "count_ok": count_ok, "chat_content": (m.get("content") or "")[:120],
           "chat_has_4": "4" in ans, "garbage": garbage(text) or garbage(m.get("content") or ""), "usage": c.get("usage")}
    out["ok"] = count_ok and out["chat_has_4"] and not out["garbage"]
    return out


E13_SPEED_REF = {"27b_dflash_w6_s16_8k": 583, "27b_dflash_w6_s16_40k": 521,     # EMS SAFE 2026-10-26 (decode lines)
                 "27b_dflash_w6_s16_xl90k": 370,
                 "note": "27B at 16 streams with DFlash; Flash-Next here runs <= 4 streams without NEXTN: not comparable 1:1"}


def _e13_garbage(t):
    t = (t or "").strip()
    return bool(t) and len(t) > 8 and max(t.count(ch) for ch in set(t)) / len(t) > 0.5


def e13_speed():
    """rev 6 (user 2026-10-26): a 10-15 min speed stage right after the smoke, before the Duck run. Recorded, never gating.
    Short band: 1 / 2 / 3 / 4 concurrent requests (as KV and the server cap allow) on ~6k-token ARC evaluation prompts;
    long band: as many ~30k-token prompts as KV holds (max 4). Each request: max_new 1024, ignore_eos. Per level: wall,
    completion tokens, aggregate tok/s (incl. prefill), the median decode-line rate and running count from the server log
    after the level started, and the max running seen on /metrics."""
    if RERUN or PROFILE != "SMOKE1":                   # rev 6b (A-L5): never in a rerun / other profile
        return {"skipped": f"profile {PROFILE} rerun {RERUN}"}
    # rev 6b (A-L2): never into the Duck window: the stage ends 2 min before (solve window + stop margin) before HARD_STOP
    _room = HARD_STOP - time.perf_counter() - float(globals().get("E12_SOLVE_WINDOW_SEC") or 15 * 60) \
        - float(globals().get("E12_STOP_BEFORE_HARD_SEC") or 180) - 120
    budget = max(0.0, min(15 * 60 if not E11_HARNESS else 90, _room))
    if budget < (120 if not E11_HARNESS else 10):
        return {"skipped": f"no room before the Duck window ({round(_room)} s)"}
    out = {"budget_sec": round(budget), "levels": {}, "long": {}, "ref_27b": E13_SPEED_REF}
    t_end = time.perf_counter() + budget
    kv = (ENGINE_INFO["e13"].get("caps") or {}).get("kv_tokens") or 0
    cap = min(SGL_SERVER_MAX_RUNNING, (ENGINE_INFO["e13"].get("caps") or {}).get("capped_running") or SGL_SERVER_MAX_RUNNING)
    try:
        allt = _load_json(ARC_ROOT / "arc-agi_evaluation_challenges.json")
    except Exception:
        allt = dict(CHALLENGES)

    def rows(g):
        return "\n".join(" ".join(str(v) for v in r) for r in g)

    def text(tk):
        s = "".join(f"\nExample {i + 1}\nInput:\n{rows(q['input'])}\nOutput:\n{rows(q['output'])}\n" for i, q in enumerate(tk["train"]))
        return s + "".join(f"\nTest input {j + 1}:\n{rows(q['input'])}\n" for j, q in enumerate(tk["test"]))

    def ids_of(content):
        return tokenizer.encode(tokenizer.apply_chat_template([{"role": "user", "content": content}], tokenize=False,
                                                              add_generation_prompt=True), add_special_tokens=False)
    ask = "Find the transformation rule of this ARC-AGI-2 task, then give the test output grids.\n"
    keys = sorted(allt)
    short = [ids_of(ask + text(allt[k])) for k in keys[:4]]
    blk, n, longp = [], 0, []
    for k in keys[4:]:
        blk.append(text(allt[k]))
        n += len(tokenizer.encode(blk[-1], add_special_tokens=False))
        if n >= (30000 if not E11_HARNESS else 3000):
            longp.append(ids_of("Reference tasks:\n" + "\n".join(blk) + "\n" + ask))
            blk, n = [], 0
            if len(longp) == 4:
                break
    out["prompt_tokens"] = {"short": [len(x) for x in short], "long": [len(x) for x in longp]}
    new = 1024 if not E11_HARNESS else 64

    def one(ids, res, i):
        body = {"input_ids": ids, "sampling_params": {"max_new_tokens": new, "ignore_eos": True}}
        req = _urlreq.Request(SGL_URL + "/generate", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
        t0 = time.perf_counter()
        try:
            with _urlreq.urlopen(req, timeout=max(30.0, t_end - time.perf_counter())) as r:
                j = json.loads(r.read())
            j = j[0] if isinstance(j, list) else j
            res[i] = {"tokens": (j.get("meta_info") or {}).get("completion_tokens") or len(j.get("output_ids") or []),
                      "sec": round(time.perf_counter() - t0, 1), "text": (j.get("text") or "")[:300],   # rev 6b (B-H2)
                      "garbage": _e13_garbage(j.get("text") or "")}
        except Exception as e:
            res[i] = {"error": repr(e)[:200], "sec": round(time.perf_counter() - t0, 1)}

    def level(prompts, tag):
        if time.perf_counter() > t_end - 30:
            return {"skipped": "speed budget"}
        off = len(E11_SERVER.log_text()) if E11_SERVER is not None else 0
        res, ths, mx = [None] * len(prompts), [], [0]
        stop = threading.Event()

        def sampler():                                    # /metrics directly (metrics_snapshot() is empty before ENGINE_READY)
            while not stop.is_set():
                try:
                    txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=5).read().decode("utf-8", "replace")
                    for v in re.findall(r"^sglang:num_running_reqs(?:\{[^}]*\})?\s+(\S+)", txt, re.M):
                        mx[0] = max(mx[0], int(float(v)))
                except Exception:
                    pass
                stop.wait(1.0)
        st = threading.Thread(target=sampler, daemon=True); st.start()
        t0 = time.perf_counter()
        for i, p in enumerate(prompts):
            th = threading.Thread(target=one, args=(p, res, i), daemon=True); th.start(); ths.append(th)
        for th in ths:
            th.join(max(1.0, t_end - time.perf_counter()))
        wall = time.perf_counter() - t0
        stop.set()
        lines = (E11_SERVER.log_text()[off:] if E11_SERVER is not None else "").splitlines()
        gen = [float(x) for l in lines if "TP0" in l and "Decode batch" in l
               for x in re.findall(r"gen throughput \(token/s\): ([\d.]+)", l)]
        run = [int(x) for l in lines if "TP0" in l and "Decode batch" in l for x in re.findall(r"#running-req: (\d+)", l)]
        tok = sum((r or {}).get("tokens") or 0 for r in res)
        o = {"streams": len(prompts), "wall_sec": round(wall, 1), "completion_tokens": tok,
             "agg_tok_s_incl_prefill": round(tok / wall, 1) if wall > 0 else None,
             "decode_line_tok_s_p50": _statistics.median(gen) if gen else None, "decode_lines": len(gen),
             "running_p50": _statistics.median(run) if run else None, "metrics_running_max": mx[0],
             "errors": [r.get("error") for r in res if r and r.get("error")], "unfinished": sum(1 for r in res if r is None),
             "garbage": sum(1 for r in res if r and r.get("garbage")), "text_sample": ((res[0] or {}).get("text") or "")[:300]}
        print(f"E13 speed {tag}: {json.dumps(o)}", flush=True)
        return o
    _w = [None]                                           # rev 6b (A-L3): one unmeasured warm-up (JIT / autotune)
    one(short[0][:2000], _w, 0) if short else None
    out["warmup"] = _w[0]
    for L in (1, 2, 3, 4):
        if L > cap or (kv and L * (max(out["prompt_tokens"]["short"]) + new + 512) > kv):
            out["levels"][str(L)] = {"skipped": f"cap {cap} / KV {kv}"}
            continue
        out["levels"][str(L)] = level(short[:L], f"short x{L}")
    if longp:
        Ll = max(0, min(len(longp), cap, kv // (max(len(x) for x in longp) + new + 512) if kv else len(longp)))
        out["long"] = level(longp[:Ll], f"long x{Ll}") if Ll else {"skipped": f"KV {kv} holds no long prompt"}
    out["sec"] = round(time.perf_counter() - (t_end - budget), 1)
    return out


def e13_bring_up():
    global E11_SERVER, _E13_ENV
    info = ENGINE_INFO["e13"]
    info["stage"] = "install"
    vpy, _E13_ENV, info["install"] = e11_install()
    _E13_ENV.update({"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
                     "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false", "SGLANG_MM_PREPROCESS_DEVICE": "cpu",
                     "OMP_NUM_THREADS": "8", "TORCH_CUDA_ARCH_LIST": "8.9"})                          # Franzen env; D6
    info["stage"] = "fa2_patch3"; info["extras"] = e13_extras(vpy)
    info["stage"] = "topk_test"; info["topk"] = e13_topk_test(vpy)
    print("E13 top-k test:", json.dumps(info["topk"], default=str)[:1500], flush=True)
    info["stage"] = "host_ram"; info["host_ram"] = e13_host_ram_guard()
    if not info["host_ram"]["ok"]:
        raise E11Fail("host RAM guard: " + json.dumps(info["host_ram"]))
    info["gpu_free_before"] = _e11_gpu_free(180)
    info["stage"] = "launch"; info["flags"] = e13_flags()
    launch = [str(vpy), "-m", "sglang.launch_server"] if not E11_HARNESS else [sys.executable, os.environ["E11_FAKE_SERVER"]]
    E11_SERVER = E11Server(launch + info["flags"], _E13_ENV, "e13")
    E11_SERVER.start()
    print("E13: starting SGLang Flash-Next:", " ".join(info["flags"]), flush=True)
    deadline = time.perf_counter() + max(300, min(55 * 60, ADMIT_DEADLINE - time.perf_counter() - 20 * 60))   # rev 6b: 45 -> 55
    last, last_t = -1, time.perf_counter()
    while True:                                          # /health only while the launcher lives; 15 min log silence = stall
        if not E11_SERVER.alive():
            raise E11Fail(f"server exited rc={E11_SERVER.proc.returncode}: " + E11_SERVER.log_text()[-3000:])
        try:
            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read(); break
        except Exception:
            pass
        sz = E11_SERVER.log_path.stat().st_size if E11_SERVER.log_path.exists() else 0
        if sz != last:
            last, last_t = sz, time.perf_counter()
        if time.perf_counter() > deadline or time.perf_counter() - last_t > 15 * 60:
            raise E11Fail("server not ready (deadline or 15 min log silence): " + E11_SERVER.log_text()[-3000:])
        time.sleep(2 if E11_HARNESS else 5)
    info["ready_sec"] = round(time.perf_counter() - E11_SERVER.t0, 1)
    info["caps"] = _e13_caps(E11_SERVER.log_text())
    print("E13 caps:", json.dumps(info["caps"]), flush=True)
    info["stage"] = "smoke"; info["smoke"] = e13_smoke()
    print("E13 smoke:", json.dumps(info["smoke"], default=str)[:800], flush=True)
    if not info["smoke"]["ok"]:
        raise E11Fail("smoke failed: " + json.dumps(info["smoke"], default=str)[:600])
    info["stage"] = "speed"
    try:                                                  # rev 6: recorded, never gating (a failure here never stops the Duck run)
        info["speed"] = e13_speed()
    except Exception as _se:
        info["speed"] = {"error": f"{type(_se).__name__}: {_se}"[:600]}
    print("E13 speed:", json.dumps(info["speed"], default=str)[:2500], flush=True)
    # rev 6b (A-H1 / B-L4): the Duck run never starts on a dead or broken server
    if not info["speed"].get("skipped"):
        info["post_speed_smoke"] = e13_smoke() if E11_SERVER is not None and E11_SERVER.alive() else {"ok": False, "dead": True}
        print("E13 post-speed smoke:", json.dumps(info["post_speed_smoke"], default=str)[:600], flush=True)
        if not info["post_speed_smoke"].get("ok"):
            raise E11Fail("engine lost or broken during the speed stage: " + json.dumps(info["post_speed_smoke"], default=str)[:400])
    kv = info["caps"].get("kv_tokens") or 0
    run = min(SGL_MAX_RUNNING, info["caps"].get("capped_running") or SGL_MAX_RUNNING)
    return max(1, min(run, kv // E13_KV_PER_STREAM))          # pre-run capacity rule: resident contexts <= KV / 60k



import math
# ==================================================================== E13 LOADPROBE (user 2026-10-27)
# rev 6b OOMed in the Marlin MoE repack (>= 20.86 GiB/rank) where E3 (0.5.19, FP8 PLE) used 19.06 GB. Two launches, the same
# flags, each with the memory trace (e13_memtrace/<arm>/trace_<pid>.jsonl: GPU memory after load_weights_only with a map of
# every CUDA tensor, per-module repack deltas, the state at an exception). Load-only: no speed stage, no Duck.
E13_MEMTRACE_SRC = __import__("zlib").decompress(_b64.b64decode("eNrtfXt320aS7//6FBhk7wmQULAkO54sHWWvYsvZ3PVrbM+ZnaPo4oAkSGIIAgwAStR4NJ99f1XVDXTjQUlOdmfP2dU5iYlGP6qrq6vr2fjCOT9+7Lx6e/bi3fu3P5w763idFzdOVUTT2PHKpIqn27LK18lfYyfPnGoZO2VcXMXFl6Xz7s8f//Xtm3dnH/8Vr9KbZ06SxUXlXCfVMt9W1HH4+vz1x/dnz8/DFz+994ODL5w/FdGmRDdFHDvzbTatkjwrnXzulIs0yhZBWVTBOp/FaZjm0SwuAvkHXTublGC6XsYERlQ5qLZNYycpnWS9yYsqnjleFgM0J54lVelEzjxJY3+MUR3nRTyPtmn1mrp+1fQcXsfJYlmVIU3AcQ6/d35890eNhEk8z4vYeeRE8wq9fo0e19GGgKVRbpznf3xx5lRxVuYFFRJuGPShATd5WW2KfBqXpR6XRmScRuvYwZwI7rKKgMVHTpzNRk6UzQjvUebEu2m8IXQ5HrV4+/a1s8mTrPJ5uFCwgY6B/fCXbZRV4TrGQswc4w+jbeLiUGHOmCqArCKaRAu+kKfOa5FkC4z0Noud//fh7ZtXjFzqTTcZO9+1V/z7R0xH4XebZPZ98JcSSA4OZLGcvNS/ypvy4CB84ZyiLIizq6TIs2ARV57b7s/1Dw4OZvHcAUALr4inWFyaluqIBjCfq2Qd8zNqovdZMq2o0cipTot8m808qhDQ/zx/5DzGf4DzFEBgcPzyfJ9bV8WNDEN/eLuOViCwovTCFyOsSlJWYb46/VhsY7+uRqvg5Js489BgE1XL4C9YK24xdwUpn4yBbgU5LkBwIxe9ZtOcEH7qbqv54beu70SlM2+goL95cF1ge3rUMpht15tSJjcTwjstq8IHzbo/Z67AJQTknGs6arrbRGWpMQuC8BRarYlrnObFdNkUzp0sV4XBdDuLggQkcxUlaTRJY92P/ivialtkzqfbunQONoDlyKsoxQIZ3QCKEMgJk2yeew1adQ/uLL5yx2aD6bYoYtA8XiRTXk83StN8atcSag/5TQSGgXoWhPTnFjFzuFlvU/2SR9jE0apVK9qF9xqEZo6mggCXMcA94d/b3tUiCojHXUwQpuKiyAs0L+JN4cX+xfjk6OjyVq9oOY0yjxmTWg/Xdc/3MjAiotl2kyYMvzO5AU/Ki2gRKwhLKsqIZYG0wR/AktLohrk0NllcApB1DiSpiaPyrLrZ0EpjhJNvnDQqFnFZqaFL4XL0bgkOiepVXAYAskuEk5twlRBbxA/VZYmTCT2UMY6FU9DWiP+7uKSiyiAdWp5QUxqNUz8cNeQIPEj/SUUnjue5m6iI1tiRjJeA5jwLuSzGvGnH0UTDGlmnL4Gd2O9Z8frPcyfb+TwuWp1K4XCPrb1EoFJD4JQgTapxZ0RszopOxzd5Fo974ZnmWZVk27jz0kK6+YflwT4NthkhfxYqsjDQbP6t4hvU9qpANmWgFgzPURWFm6rQHLb9l03QDvUyJoWe3odZWWv8kfTVhgE8t8IiYSbZdh2nnu98hd9xGq+JhxBJ9QwKfNKMgG0mNiyA1atzeupgN1aR+0BkU29BNJt56L13VHuY32EYIuaBYQzS/voUs38YLMYu6W+NDQ98uoErB5r7hUsQbsD3Z8kiwYZzYpCrs2EC3RCyiEqDEtSMAz1we1Z8hQ7n7ifaeLfjTxjg9m+fcHjRmtGE/QBsjUQ/70thtF+OnC+/9G/dTkeKO1ysLtGjemBBYjVyjugs7JnPjCmtZzBXBsMudV2/byiufzFTY/ETDzYbHox5VRBtIBfMPC+bjNQWTiFDAIJyGQECMF8DSdIEXJJEFzDtMm5JGvVBIEvHHDXkfYMDoVlOTEMow67QUEvPGaVQiGosOpUsY3sasZA91iUdgyDb0zRaT2aRs7oaO4erq4vjSxxCT48u/f5eGVnotuHiwPYGBRefXA1ZeXEEHu4SgvjpmJ4YQ/x4Qo+6n/Li8eUtU1zJ25NwhkPwm8vLh52ldFJ2D9PH6jCdphCTnPAPr98V+U7xRz5gIaUkVRh6ZZzOR84va72sEOSr0uDb9D4I6b38UrX4N9cFKbVaHxijgLhw2hb1QLQAfmcWqpbXDMb1mo72Sviqa1EQjM4jgCai4cHgGVHQVlKjBvtHUf0fDPHz1gLRH4v8n1yW+9zNdZSqhQIZSG9EBwZa3V/WJFCBQGpU+EHIL8MQrzvL/ATL3Hdsu6IGompE9Jhf45eg4tbmC0WUlA1HnXQxRrzGm4hqI8KpT6cIuMWh40U95S3KYYq4cDP3kpjzcf9b1uSkxsyU06NJ6c1853s0fOp8951zckRjdDr+7tR5fAfiPxvlAhq4SS+atbhuI6iRslU5P1GxkqBVMT+ZK6K2Q6GlYJwr0yURnqLqV1gMPAVdPZ1f50WyCNPrfCS/Nhv1Y339yxotX3WtByMU9mj4vJmCYfX8QMQqSHokvrosEmJaJLQRw6OVweMnLM7YOTKQeHR72+zpDjCekuNrICoSubV6ZPIkGvlCDXspOIEFo3fxO6OEbKoAPXz1VWdHdNmDLIhG7B0QNjJ5koEUWn31yqe9GmrdotHRyptsuoSNoSvl3S1WsqY8uDu6CMJBb6CHFhRHDDEQQym7NZlzl4A0on6LFTS7f9DakXFG9kE/OG1O/kNUxvfn5gZUNVPv588WKtXeUAiQx8tBHA+z6mFgOqv3eUM2vEiv8yA7sBR11oXAPUkw+ASdBce9k4nqB7ZCoo6pQao+sZKWQivFJJdQg5qJdQaiv5skTmd1Cy3qeFooYYOVl8z0+Y2J/wvxYhsdipx7WCTL2lGVTNVcOxV81bLPUtpq21NFWu9nt7I/9rBimxjo0Ihn1onH/atDjbbuAQ7X8IVliJR/0mQSRJNp/4t1NF2SufxGkKVky5cQrOPCs9oHryHRvYMJUV4aK/bx7P2P5x9JHdtvPHcPGhEEtAeOCs19E0+VsCeHNxkp9a4+pdOnRRpk6iOLz++UkCeDdzmkonXqwXpHAwLUHgwEzeSCBrYGKr8NB3elDBssxOC59hMM2TsUXFTXescbIt4xcEZHARUpMjqwGhAKjZdeR1bu9NwRd+80szTSSm+7e4jL/byNer2DxbY45QBKgCwTQUOIFuHmBjbzmCw+ZAdPMhhQKw+ijKZ2aLtwLBzij31Rb87eiCsKJpFE3EOeqMwwRYonZr0hjsxm9GDDFpnah2J5rO7lkYJa7Zw43snRydPD46PDx4BMgxC+/Onfw1c/ffjovDw+fPn7Ef45of89pv99A2ZcXefwonx0oi17nL7mAhjEsqopyq/JfROlN2VS0njHgbOYYhPuxhpwYmWHhGNaiWgKa8ViGkzzNI2nZFBpuaEixcDggNurZImnrLZCs5fIdPksivgaXaFneNMyxs2Pz8PXP/3geMqFAFPtUx8YzaYxwwkeVTngh9PVM6P66RGIQUATt5iiENgrQN9OVKwZzSeB8yZ6g1nQxOADRHcEWF6M2HkoHhPqVSP/9BjrDrcX0FlgaPgKl37gfPjxFTidcxR8E5wcO1NMr5RDMCy2GRZb+F8AjF1HxcwLgkD8YzBhwDtSYSCPe9TVNzfjk6ffPHr8+2M6x3KeJ2wRToq5pMRNAAJItoqAhEPVq3NdEEEWjscezDxPy+Ca/ZqrOGbvJtHhIouwEWIZ3QIRYz75/WMsdjQrMSp5UCJH973M81XgvCTrIrqZw89UabSEL9++/9PZ+xcfpDvdBLIzAVNE2YoN6apY/J7Si+qbmmEFJsQGRmq1ZjFcTfAJigEfu2SDjXKxTkjCiHaQbbJ4RyaiVZwRL1EOSvzfm5PZAzxa14UZv1jQv2rgfHN4TC+3Jf8+cRbRhmSFmT8y5pbl2aH0pAlHYCiFIkDEqFuYHt8vycgjNWJQb5QSIuGtjRSNe8R4sbWueINsogU7MeByzreLJXmR8U67nh2avAXP33Z/c76vMX72w4fXZ/9eg+RFOPzI/A9oyWiHljClEnVUamzMlwzBQIOvWzXujT98OCMXCTgnuUqqeFFjvOSWCo1A2xT+yQrkTu5WUV0ivMdbXmrlsvGyXEadb9P0kKxeeEPna1Qk5BOh45GUHf+ZM99A68fKz/mf+ebxiXLZCxEQ0TmTHFgExS+2oBVyptPuoyOCCrH0i3hG1HDFtAb5eebXztx6H9BJFM4rXb4Gx+ci+gEnLzjGh4/n70gOyCqvz90rPIVOJzAfWIvFVEEtWT0mFtRox1onJuV/xjqx1vYX05AZlQf6KWKRcdQRTaYsTyT6IZMLSXQ/Pr+Q0S5rUYOWUU+hOW3NmmCELaHf6E+mTp1EMPiYzWB06fZrthdsLkRyMQ+HA21qumNGPJplOOISQRyXgrboUCbQJn4H6ryQN237Ud2vbTeypA5aC4GXlpUXBMVqYRq5WtmQGPbQsrlJER8xZPBRC6+HvjVmWC/DBIQA9YXUULLDKUWmY8T98bmnavnjX2cgbShO2wtNK6+hCgo4NGTwgIH6xhGEDAzTWL/0XA08/Picyj98fP/H5x8h45y/wpECEnKXyWwGNs/qXEmLZRWQcpvSvMhYvwqFcdWPahL1czLj32C6Mik6OZr4DeX59XJyNW+q5emRHcrRGHFAhUmpT2CqL+zwI3dgLICorrnY+1O04n6d75wnvOXsPjxi3ugKTtbY9KySvrxjNblPL54X+boBfadAh4h37N9vWHKj9I4WXEXpllT332jYnHgWyRU136KHFg7AghGNAnKr6FSGHDYBB29jY07w2aQy7moZXzjvn0Ncq0Vh53hEEq5z5pOtpNhOSRKaOa/zc+CgZDXEySd/AUuwrWk7kKF2YABE0D9rom0FcGdOr6v33I02TYWzuJx6VRMWgVmA5ajzFXIGO5hwYLLfif6tioRFKHagLrZJdcPYbeQlx/vaFGnmabQQPBKey+Zs50KsQOnXkQ4d0p/xkVc7ue70UhoOspZH0RXAjRf87FmBCq5Mikz4FMajpphva4sSu6K1w5xlGgGHQyUEEJ7m8VO9SyetZ358fGLQ2JrwtquDf5SUw/76CgYCM0KgVRWUDC54sc6kV7LPrXf696UfVDnP1XC/BNsNIg9ikrRO17Jep+sdCeCZLNcp0ZTHwgowIGXeOpPJtot3vm/sPxsxHBRFSGRwwM1DDtVrv53mazCgXQubv9MzpP34WZiyZkrjAknkBaT15llz2a4py+LFMRd6FUmvh8d+UG5JnFA1/APjUJnp7cPcJE1x9GQhSZClZVNU5EwSOg7cIu6n8Wx+vUcaNJUeovHjkyMdQTBJFhQ3wMs91FSkd24YP9ENX77igJCHkKsIGGUxEKL44c9vnofvzyFw9bJF4YjQoorI+WHsIEogcFjbKn8OTn4O1unmZ5hWINdX5T8pg0k2BcLowC7igIgEoZYexvdZJgMcHGPBJrbGP0ytzAOkY1y60wXSb1YaimP6QukCaVyxcjNLokUGS0YydSbQyFYqoCtfDRv+eaLD5qiTxhz12VZwRnQXmYXr/f+//Rz4eh1+nn39TyrMg0OcPrnza6VcwDgySzkkgYLKXKie5I+71QCXqlpOMimFZsi4iEfkcLRmgVhr9qp/DO+DcozGD2RvbWlSgBixpl1PiwIiK5PuMPNgmsZRYQXfFZolNcyW0IldzP+yFs2/lH7Mv2V1TgnZJNlTmK1oftXRKaIumJcgHMKlEF7XV5G8hjeNNVlvnhnQcbH3VURunVWPiRsYKqsLWs3L+9u2e+24qvI8q0c7+AwbLgGj6Kzl9TcN4malrge/d+/Rztyz9x53TcF9s9dBJ4Lz/8sINxQpto6zXQiiz+qa/jVwriDHTgPUdBLYYONooa3IlFzLcdwFhVfZxarvlu0andN2vbwQl5VPmuGFiF4cNckjSICdlmNevrpE5NKl1U9GHv+2/0s5vzoEtMb5MCvJduW5YO2as7uyy1JEhNNm4LNjncc4Q12KWCVDh/PtWJ8ZY2WXIYGZbV0jlhXxv2m6hW2mJ26vCLAPleHW7voiW/PEa3HXRsDF+NvLy/tPYzxwwI0FUEc2dwa91vnhHGf3uVhQDyV7gyaE8NaCQ9cxn6qH4K2JoNMQ/CbcsEZNszHWMspuPGGrOx8RYY1C1V35rhNlvtuQtpHmEwTYastJWEyPQ2gupK/wmL0BmWjav8PwwsvWrLXbVD+qKb3rzVKnfVtjU+XARgSWj2798ZBIIU74w2vSTagZhqb8Bh3Q3fbQiDTD4kL/vsUp7vVMAap8a/cOgW7t6t7Z9Owmtx3jrGGsR8w27AVWawsv8aZpfmFgNpnQ2kkMMS9rh4rstraSabIlmLs6fMkwzGnR5B78AeJ+MqMT3FUnnFsHx6b5dVzYh3VI1jFU5uBvSyFPVMR3TPoGGaIbJkjr0z3SGs0Ekv1Rl257Y4HbTLHbzBAGlGuehJu+3aJFMHPt/YEY6W1BRjw69bF3qSJtc7eWGTjYmEZkAfhCsngIEkreUU87errsAwQN+4c1EE4SXG8dmogw7wsDnkvh24PR/gnZZjvyQo/Q0e533CNvN04KMns4ys2RF4ab41+G8w40NGpzkF1N/TJ2L+N1kK1iG/mDA7RnoOOcP3VMvMqgwYGCtFPMGEEVn4ytghrJaO90ev8GTSAPsKN8xqjikSImQ3CTjRqovQpJEM7CGgGKvlmUddvyVW1LICWAwjUhmo84bpOEcTilSOU1hIgLFwVCgU/3bCeutGcxbITD2j4pAQDJgrKf+oAZwYg2CL04wHbKpqGZz/fOsQ08i/x3g6+qPWACbEulQJ1syCDWAp3BVvz5s6039bw0ex+a2QO0PgsRdb8PQMV/As27gJd6b1mVXKGZtmHJJcuSKh+wLd1/5LvWUrFSzc1t1NtbTx3Yzcm62y8Ltmw7SkivxXNYhMnRQVEXj7QPXbV0vNdRkYo7uoiR4oXcTlvYmCuqI1FgSOi2c792KjqqX0YzZc1apmOfv6RTDmljeMeWD55RyNhlq4eF7kZ36NPLYLPROReWg4PrchWtucnwtd+jPcO14QjYkYdXIhiIk5IE1+sVQKO9bgGaoLZCmP1Rpqw3R5RKk6R0cUjpJoIz8qufmrA0xRqS7o4t07I1gzL+JQSPgBdtu+6dgQbxwq56qcyj6NFvOcNQZLlxfN8wDHZkHLLJtfV2Gq9XBRB/nSQGeLbcny5MJw2Rm9sJEXF7ZGo2PhnQp4uWL0/Y6MIUO3WROkQG8onpqC0RcwViZXn59mCfkJoubG+s2Zykv/ahgeFrGaIRb8bU50hzQz5qhAOud7c9DKkFeE5Gb5qqYvngyTyGdwjHmR6QiNBHFpS267WMuZv6CCF/q4c+R85Jy2KjyV0Cc9jCj8qBStOlvmtOjfIQATqnMhhXk82JWvD8G8+A6g7ejRgf1Q+ggrFy1vYkdIOiSZjVses6ektRJuJJJLyKA3v8mkZ1jNdXwtBgYesxMJkGPY4VvVuzY1g6vRpw24nFk4TiDQbYa3NvwLZSMfT7ep4yPYqBjrwqaop0RLM9WkNPrSd0wF+T/6WshbVBoyhWkA/A6ogaP+4unKs27ri79Wlw0jpCk/rb0n5dacIexEby7I4kEt3YEu9GjVo8bok7eKU0wrGtH1r7rG1U67CK6YV+RRyVO9LPzT0I6x4pADKeHDv97p0xq2XvXp1LdF21xBmk4+0QkudsKcYVwXLsCE5Smydu1j1g1gMSoBszgay+9uLANHEyOVum4eVqQJqRCBNNVb49D5E2VEIOQDmkZAYV2alndGRCv1zdIytnudIZI/0WsHvawe9hu3Y7M3RciBuNO+lby6R9l5mHhJelNiZqJ1CPU2AZSMq+1zU+50JsthFnEFMU0mnxvwG3G9GbhDHpetD9E3UtjD5eHTqekwmCb5xrYDTfFuVDc51sWIR5DfvyaidwqC5/cBuxrZSQJY7Ra1iYsYyleqGdGN3cGTAidVBIHsmQyxc30/zZ3qmKuhk2aZ1nOgcEUWg5BCY24a051wOBmIaD1kE8uF8P23Ujcs4KURL3awtfnHvTeCdrv6R1PNUiq0T0krjj6aGMDAdV1G89VbbTFsezqFarjGvQ6gJCTbNFyALE6+ZtKByPc2LkXJI8c/+zOpUOc4qxGuixTVz6yN9DVyrcwEzH4fMKircFl+88euScGNRHCvL8ujZthGmyTugAwvlknk4qB12bpRDIYOf5SFIGTDwSU8hPQ9mlRl0uvU/K6WbDiaU9iU2/Kj+QZNw70/baAZvQPNZRcUOYVrGblIhhRF3qqNv6JgEznpQDjuKN+Y7DSG3uO7CFOfjfZWu1e+ze44QZjEP51edMQ3v3zpmxpdrPz2QzM2LY43PfZJivqQCAo8fJg7JjMGI3P8Yx02JkvDo/RxJnyNTRGKYpxaNmecJDg5PAsGVwkOAfxCQC8wnlCtShcjldgMZ2VJ/vPEtTTmoon9V5NkWeV4fTaEtGasmv8Qg5FGpIuprz/Pjw+be+yrI5t+QZjrcvKGDPo2miWfjh+dmb02O41WRixjo4nk5poakpA44k54iaKiduSzYa0U8j30Mj4JMe8NXZn8/ff7g18aE9vSqkvzm1Pb5liBKI1M1A+KUkNFxCBy1H36nEuqLKuZCwvpERnvhMRR6OzahEjkAcacRTysIjlcChnIaUmpM6esk492KZpxS4bvTC6uAzGZi802Px9FZPnzhgH3RdXubs8gJZM2jJAgAVIwAE0gudayvaj+pOQeQlgdg+1v5KxF5yZqajsjZhX+PcGebOKp2olrnJYkYJLZQHwvkRekU4BJSJU0vfTOJoGL58dfZjk+j0qFpvHsV4pWVvX2VP8FWHKpPGJQHI5QmwII99LNlIALYE2g+B/ThaU9jDPAZAzyTAySX+q4o40UTtR3bWSguc9EZE1kxnfKGayl2SHB0RSggTJfW9SjYuB/XU3YsISHiRmnAJLYlyFJLXEUuzNCSQ/aKJ1yLxgpD6OGD8vDoJ37wNX5+9f/UT9sfYOdtW+XvSL5+Tvrcgbm3n1ArTk8QnIhC1YWincwYnbKB5/Kc3Z8dPlf0Q9RRxyQRz5SV0JC3FIWGENjWZQyUPyO1jJm7fNIydRhYSCvEmGxl8WUbAdc/R0zQjPn80Oh6djB67tVVw5LKgt+PQ2Q1Z6Rs66hdH9WvqrkNgrmr/4fz8TX3xg8psqbNYdOXeC/maKwbrke53cSApufOANgt0QzUbEjKFvs0Lc95+OKfjzohYI29sU40FWISr1jORawhMLanzimSggcx7jRoR79iFc2tFnq41YkjTE45ZB9T9N46e5ls7SN7il9ZFZC5bNrlc2TjrUGvxM6nSzjU/M3PSTZrVgY6ib9xJEi0rQutOSHiWrD2+CuZEdJZdbX88pgh1hfP8umTLFjJVxB55dKlfhHRVGr0lzxXahHzhD+6Gaiofjy99HVAZb5qq3jFnD52wlF535fdqun9/8tpRl7KVzNiny222GptZbiRK3cDR9vcnR87rHw46QUkqX02mx4wHKl8a1yl0cqI1q8Vs/zrfpsSFcVBuEvBjThhMU+TRlbiUL2bUKFauepaYT4KODLtyGVZxNBbsZRF20TVlVxVHnC7pEUpGguFDFPqNAbjvbbMmsnq/NuaeLdfkJZN5aCN5GdGFZHNKYCRZhgIwJdpS/n8UoNC+K7E4IjAgo+EKwKORQgtB39JFyazJyPGKI1sql9EE0Dr2dGrXkRv8/o6X4jk87nh+iL1nE9ut2BXtyQnw9Wlv5b66jB1qoNAMOvlrXORo6wOndO9l5pE7C1PqxtvS31U9M851o3xZSGZNxK68oz5Lj56FMZxOFYdoQZXNlSMI3ewBnhjHvGfubF1XBMFmaps46OLSTqM0ryPs754HbRCeBmnV3bkERNjD+F4mnzPU4dBYYDXDY1GAAs1NYhW0lYUZIQFJ2yHNe5oRWwWc1GzXaobxaActkx5PYDQhIpIpXAUUQdFygD+TTUf3Lc4P2gbBKYvpdI3lyLlqopgt2ztzaeuyL3plE8Fg8suaAlQA1SkB+ohbWlp6x3mFsfXu0EPzA2KGTy6HaoZiCSVbjS4ykmfoqNvtDbjYdQMudvvTZeh04gPs6MBaeF5Gy0Oaz+edW2Dvy9lWJmczvHjEEhQls0Zk08V1k4ESyRgrfdDX5LxS5DwSPe/U7IxulZjPfef/OMdHR/9MKW22lU6zudUeRnFd1/JWuP30ukWUPacx9GM2QRyKRo1zES3gk1F2U5zP6y3ZNWCkOLDZyEj2tx3w0oqH58UhcNK8hmHPDrVrtrraqa6WidXVwK61a9pdEW0ATfXydPYnLuu47uZdATGnIJhr+pePWGvT9WdTkUzbXB814us7hmRbVCn1jea9yS/KaM1OR3XfNgePcjZno+mMO1Gs8OTjZlLJrmEZY+7atozEMl+4dJ+5rpAhhmAhMvxw/bujfeZuq6u9HfLSZsroVPYF8/LUeu8FAn4uSDFIOJocOlhSlsR96riqJp9nMFp2LUbdUqJVW5EFeyMfrbhHU5/WNXSQxbr9XoedNB0ctCKlsCPXcV8/6k2nBztoZ8VLT8pJ2Cw0P7JlqF04h0FuRhFDSPI135GDFWZgMuIZjxI503cFJFWo7PqVrs/m+ZDLyZRXgNwTbDV+G2JEupW7Yv5kJq5T0xbLvjLCV4CR1UA0kRGpcnWfZOl64eX64SuTAhMyZhvbDppUy++LoJ9iykek8IFiKsEEa8OyLAYwC3hXCvVykhZvviUE2jVMqFSHgzFTAEo1JriYkuS5oZom2RDqVMNq6gvKR0ao2Lrv9nT44cv6inMdG7buXIluVWu5BDDyRcYQtse+orGvIvSw9vV1xb3J64ZdVdLn+coZuSmmbZgt9xNKN6Ipk9x76alLLw38DcnEa00IAhly8+DA8QcTMa1LkSQgAXd20LdK0uQqru1/OvVe+ZsU/PGesDkGzm3aBeb2J6g2YJdMHvG6j4GAH2zxqQvIRbCygiV0sxMM7PVsSrXNBneZ2cTYKRcqElYE1AGiVNdZU01NHP49vE1m57JutjXqc5xO3T4/DWTDHvQfXUXXPLg32/eOT4F0THNaNlEXuGRyf0tmhuYAIPGS8p1pn2d+bZpyzvboZK/5lWq//NOe7HHdpZlBfuLqph/f/tv5mw/3aC4VqTFdSGSKa+burF2/uM6B/Bh0iMExIFE49OEd06dCe7N9l9kH7IIZvBYv6t0md03esft4TMIwrnS6MA7sS75rQnJ+ef9jkxNMz+BPStLa8aIuGHtXMweY1MFn0wlcXeP6rjnDL/VI+aP2X1tRb0yKUDYYiHUOxWaUL9Xr5S76bKNLTrq5KrUmZ/OSeD14UYySuxXDUk8i914J71Inh5xFWIG9p5E+tOScoWwWzUwO+nJ93GZ2nAC2al8T0z5DaolWba13yKiUxPKGDo2ExeHI8Pm1dc+JNpDCxHnjSOQQ30Omkjab1EH2N4EoSMajgPc17K0oar7IBeLjHxzeIW4JXGdH1R0RCRXLoXk3l+yRt1XuHZuytRVfweByFT2vvNYSXS/MsU1x9c0SJ4M3B+3Z1h//yDc2/I6jDEhwQqIiLOKKq/RewyWncnyiw3AKV18f4OH+AL91lYOrbh0z7flrR259gYizQMTmBrZMX4sHBvfrvwSMJayHZ0OTXm3wMQk1SaaGoRpUgCzg7LIJR4bxwp1utmC/7Kr1/PpjMlaeZ9V4Eo5b21HT6oUiSFCfnGtz8nZceASBR+Rg+VqM9AKaLmch9piFOEV7vybpcoTP/cfZn/PTezULoHhyFxSmHjJucUXzndrkfbv6jlyP9rY2LmHCNaUVfWbpB0i9NF8V09Y9bWu/f3PdZPvsNC6bLKMr8U0327ps3PYyK521Thc/QhaEteKRvund8RqvCjOCQq4JEUyr7e91ITHIl1MNAMlz3GbKncF/tEm34oBWB29z0HK/IolyIEDNgrDm8YQ5UESOHDIv0gnHIR9gdDp6QPl2EL/AJIG+2bGtpvM/nCtJnrXe7ZxW3ex487BP+D6I2g5zj3yvL1RoAx9RoGCKByKxBMjkNaJNgGszowKBpWaSePnZl0ZkRz28kmfZ5ZZHvym7bFDZcMjfmj/WLPG/liE+lAdyZGG+1nfxh7RGrig+y1JiaWTbWDYeeZ/MahumuuaD4qNYVClwQf2cPpCgErnLJYcYnkjMLXo2LxRfUu4NvV6WJIDVjWG4oXQYoWXVX72Qtc21bF0zYiXoDy/a4GnNN/saTovB7zKSrqDk9BlxIXpsNq30QmUcwUPPQqlcRPmvcCclUUohSF47leweGuN/vlBBIUG0Q9RHoZRRlTNjOL6XDYdjnmlNW/SRq8QgNUVbNRXxs6Rrc6xnKxWWjI2cBVnTU1iTE/bTpWSumOmSoDeqGVqUJ1WtjiUub6BroiyOsaVKd3Uula3OQWk0SRJ+iCBBtBKmTsRZ49RuIp+ucz6tWnt05T/UJLzP7FubcR+eY+/sNza3bU0t5vJZ491lafaHTFiMuJqhifVYLMa3twfDJj4dKb7HLIc9cGEqkWJ827PLDu752dqNhLG1P1ardctXnxrp4DacX3+aX9+G+Hit/RnbTaVYsbqnDKIiRDk42dTFER2rks6v5vwxdpVoGCh0hna4t/G1nUknmDdiyuD2P7BXsdlCjD+jpGeXcRWrjCqp/cEv+fflrWUMMmz2ewTyTZHkHIVMcccsriIUN6HQbKToQEBVn2FtBb+K1wbf415GuBabP4uBaM8vHjss/RSdO0SXUbnENzasT49k9EWGqt8vCFLN1QdPW46mO31QoO7GqbTH02TnveT9DrmOd61jgpUoGJ4KkZ1gjSjHYwdF3jK7UOgQx0jOiT4mrq/iHjtuwJXSToUG5xIMSFeQjzUyAynwEP6zjHf42mVcVvIVYA7Uli8ZhrIg/F0yjrL2JhKCeGrWGr5uAVks6Xy4E9lVVMc1vvJzDyuzMcWOYfn4qTYsm7lWDWFbKXXaOz2/DmfNR2L2KDUUZF9nc8j3CaQpFR0bgaJt/7drjRseuUYODvIb7puCk0mP/+0zcO6LwvskxAyg0gDd/fXJMXqQf0BujLHg4S/XcfbEWPZ3HevzH6gGkkEQifyKT5GWWvrOFP3pwMWG5+bv5ItfOGh1gL3pS+4gg+KXB76w04EhsPvlYecUYu928GWmx79rtbNuFzXK1eXx6kMfegvxo29Hcxsx5nZeIlEbpxiMhy6BvFN91yzDl1DZ4WyEg/Z35vbB3wMmZ1kQazE/fgr01/gIZZzGlzDk8JS79FvNBq7Ltcm5vTRCyeZa7Seau74yZpB8hHwMdTlTTfbPO2TfytpoU/3z7gncAS4qBgm61ft9Cfh5J3nEuK219UatIkt+rIND9LKlCCkzLs3pTxKxl5i/38nrDAcT0moXm+oXNS4396wBR2xywginLlVs8c0OxtIT3JJA53fKt2+60onom9yb8Y1U8+uo/Vet/rI+6Luaog8vgsYueoUs7vjMnbnWD/jaHR9O8h24DyxatD9DVwbMnUnFoSxQg2Hf7hMbzPyjnnPPHPbCHFOtPs81+StHIga0V0LeLBJd09pB1tf30O/DPsD3+V/Tm7P8YkyEZ29/w0Nfa5Td+VG7//3Y3q/52N48+80/tMec3eac7VDAf/TH9xpi9/2D/wDb18Hr")).decode()


def _e13_link(src, dst):
    try:
        os.symlink(src, dst)
    except OSError:
        if not E11_HARNESS:
            raise
        __import__("shutil").copyfile(src, dst)           # local harness on Windows: no symlink privilege


def e13_memtrace_summary(d):
    """Per process: the patch marker, GPU memory after load_weights_only (+ the largest CUDA tensor groups), the repack
    count / delta, the exception point. GiB, 2 decimals."""
    g = lambda b: round(b / 2 ** 30, 2) if isinstance(b, (int, float)) else None
    out = {}
    for f in sorted(Path(d).glob("trace_*.jsonl")):
        recs = []
        for line in f.read_text(encoding="utf-8", errors="replace").splitlines():
            try:
                recs.append(json.loads(line))
            except ValueError:
                pass
        ev = {r.get("ev"): r for r in recs}
        if not any(k in ev for k in ("load_weights_only_end", "postprocess_error", "postprocess_end")):
            out[f.stem] = {"events": sorted(ev)}; continue
        s = {"dev": (ev.get("load_weights_only_end") or {}).get("dev"), "events": sorted(ev)}
        lw = ev.get("load_weights_only_end") or {}
        sc = lw.get("scan") or {}
        s["after_load"] = {"alloc_gib": g(lw.get("alloc")), "reserved_gib": g(lw.get("reserved")), "free_gib": g(lw.get("free")),
                           "cuda_tensor_gib": g(sc.get("cuda_tensor_bytes")), "host_tensor_gib": g(sc.get("host_tensor_bytes")),
                           "by_dtype_gib": {k: g(v) for k, v in (sc.get("by_dtype") or {}).items()},
                           "top_groups_gib": {k: g(v) for k, v in list((sc.get("by_kind") or {}).items())[:12]}}
        end = ev.get("postprocess_error") or ev.get("postprocess_end") or {}
        s["postprocess"] = {"ok": "postprocess_end" in ev, "error": end.get("error"), "alloc_gib": g(end.get("alloc")),
                            "peak_gib": g(end.get("peak")), "free_gib": g(end.get("free")), "stats": end.get("stats"),
                            "cuda_tensor_gib": g((end.get("scan") or {}).get("cuda_tensor_bytes"))}
        pe = ev.get("pwal_error")
        if pe:
            s["failed_module"] = {"module": pe.get("module"), "qm": pe.get("qm"),
                                  "alloc_before_gib": g((pe.get("before") or {}).get("alloc"))}
        big = sorted((r for r in recs if r.get("ev") == "pwal"), key=lambda r: -abs(r.get("delta") or 0))[:5]
        s["largest_repack_deltas"] = [{"module": r.get("module"), "delta_mib": round((r.get("delta") or 0) / 2 ** 20, 1)} for r in big]
        gcs = [r for r in recs if r.get("ev") == "gc_collect"]          # NANPROBE: the gc fix at work
        s["gc"] = {"collections": len(gcs), "freed_gib": g(sum(max(0, (r.get("alloc_before") or 0) - (r.get("alloc_after") or 0))
                                                                  for r in gcs))}
        nf = [r for r in recs if r.get("ev") == "nan_forward"]           # NANPROBE rev 2 (fix list F15): per rank
        if nf or "nan_hooks_installed" in ev:
            lg_bad = lambda r: bool((r.get("logits") or {}).get("nonfinite"))
            fb = next((r for r in nf if r.get("first_nonfinite") or lg_bad(r)), None)
            fc = next((r for r in nf if r.get("const")), None)
            first = (fb.get("first_nonfinite") or [{}])[0] if fb else {}
            s["nan"] = {"hooks": ev.get("nan_hooks_installed"), "removed": ev.get("nan_hooks_removed"),
                        "hook_errors": [r.get("error") for r in recs if r.get("ev") == "hook_error"][:3],
                        "hooks_error": (ev.get("nan_hooks_error") or {}).get("error"),
                        "forwards_logged": len(nf), "forwards_with_nonfinite": sum(1 for r in nf if r.get("first_nonfinite") or lg_bad(r)),
                        "coverage": ("INCONCLUSIVE_NO_HOOK_COVERAGE" if "nan_hooks_installed" in ev and not nf else "OK"),
                        "first_nonfinite_forward": ({k: fb.get(k) for k in ("fw", "forward_mode", "batch_size", "seq_lens_sum",
                                                                            "input_shapes", "first_nonfinite", "logits", "layers")}
                                                    if fb else None),
                        "origin": ({"module": first.get("module"), "type": first.get("type"),
                                    "verdict": ("ORIGIN (inputs finite)" if first.get("inputs_bad") is False else
                                                f"PROPAGATED: bad between {first.get('prev_clean_module')} and {first.get('module')}"
                                                if first.get("inputs_bad") else "UNKNOWN")} if first else
                                   ({"module": "logits", "verdict": "logits only (hidden states finite)"} if fb else None)),
                        "first_const_forward": ({k: fc.get(k) for k in ("fw", "forward_mode", "const", "logits")} if fc else None),
                        "first_big": next((r.get("first_big") for r in nf if r.get("first_big")), None),
                        "indexer_first": next((r.get("indexer") for r in nf if r.get("indexer")), None),
                        "logits_first_forwards": [{"fw": r.get("fw"), "mode": r.get("forward_mode"), **(r.get("logits") or {})}
                                                  for r in nf[:12]]}
        if any(str(r.get("ev", "")).startswith("rc1_") for r in recs):    # RC1: weight scans, PLE modes, MoE inputs
            s["rc1"] = e13_rc1_summary(recs)
        out[f.stem] = s
    return out


def _e13_post(path, body, deadline, cap):
    """POST with the timeout clamped to the arm deadline (fix list F10); raises TimeoutError when no time is left."""
    left = min(cap, deadline - time.perf_counter())
    if left < 20:
        raise TimeoutError(f"arm deadline: {round(left)} s left")
    req = _urlreq.Request(SGL_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=left) as r:
        return json.loads(r.read())


E13_PLE_FLAG_PATH = Path(os.environ.get("E13_PLE_FLAG", "/tmp/e13_ple_mode"))
_E13_PLE_PROMPT = ("1, 2, 3, 4, 5, 6, 7, 8,", "9, 10, 11")   # RC1 audit: the same token ids in every PLE mode


def _e13_set_ple_mode(mode):
    """RC1: the flag file every server process reads per forward (mp_rc1._ple_mode); 'orig' = no file."""
    try:
        if mode == "orig":
            E13_PLE_FLAG_PATH.unlink(missing_ok=True)
        else:
            E13_PLE_FLAG_PATH.write_text(mode, encoding="utf-8")
    except OSError as e:
        print("E13 RC1: PLE flag write failed:", e, flush=True)


def e13_ple_mode_runs(deadline, modes):
    """RC1 (GPT audits: PLE A/B in one server): per mode, flush the radix cache (fail closed: no prompt without a
    successful flush, so the same prompt runs cold and no NaN-poisoned mamba state is reused), set the flag file, then
    the same count prompt in every mode (token-identical input, so routing differs only by the PLE mode). Records the text, the first output ids, the
    expected continuation; the trace tags every forward with the mode it ran under."""
    runs = []
    for mode in modes:
        prompt, want = _E13_PLE_PROMPT
        r = {"mode": mode, "prompt": prompt, "t_start": round(time.time(), 3)}
        try:
            try:                                            # RC1 audit H3: fail closed, no prompt without a flushed cache
                left = min(60.0, deadline - time.perf_counter())
                if left <= 5:
                    raise TimeoutError("no time left for the cache flush")
                with _urlreq.urlopen(_urlreq.Request(SGL_URL + "/flush_cache", data=b"", method="POST"), timeout=left) as fr:
                    fr.read()
                r["flush"] = "ok"
            except Exception as fe:
                r.update(flush=f"{type(fe).__name__}: {fe}"[:160], error=f"cache isolation failed: {type(fe).__name__}: {fe}"[:300],
                         valid_comparison=False, ok=False, t_end=round(time.time(), 3))
                runs.append(r)
                print(f"E13 RC1 PLE mode {mode}: SKIPPED ({r['error']})", flush=True)
                continue
            r["valid_comparison"] = True
            _e13_set_ple_mode(mode)
            time.sleep(1)
            g = _e13_post("/generate", {"text": prompt, "sampling_params": {"max_new_tokens": 32, "temperature": 0}}, deadline, 600)
            text = g.get("text") or ""
            r.update(count_text=text[:80], count_ok=text.strip().startswith(want), garbage=_e13_garbage(text),
                     count_output_ids=(g.get("output_ids") or [])[:16])
        except Exception as e:
            r["error"] = f"{type(e).__name__}: {e}"[:400]
        r["t_end"] = round(time.time(), 3)
        r["ok"] = bool(r.get("count_ok")) and not r.get("garbage") and not r.get("error")
        runs.append(r)
        print(f"E13 RC1 PLE mode {mode}:", json.dumps({k: r.get(k) for k in ("ok", "count_text", "error", "flush")}), flush=True)
    _e13_set_ple_mode("orig")
    return runs


def e13_rc1_summary(recs):
    """RC1: the weight scans (after the repack / after forward 0) per layer and tensor, what changed between them, the
    NaN result per PLE mode, the MoE inputs of the first forward, and the patch events."""
    scans = {r.get("when"): r.get("layers") or {} for r in recs if r.get("ev") == "rc1_scan"}
    keep = ("dtype", "shape", "stride", "contig", "nonfinite", "bad_rows_n", "bad_rows", "min", "max", "mean_abs", "sum", "wsum")
    comp = {w: {L: {"type": v.get("type"), "quant_method": v.get("quant_method"), "scheme": v.get("scheme"),
                    "runner": v.get("runner"),
                    "meta": {k: v.get(k) for k in ("num_experts", "num_local_experts", "moe_ep_rank", "moe_ep_size", "top_k",
                                                   "intermediate_size_per_partition") if k in v},
                    "tensors": {n: {k: (t.get(k)[:8] if k == "bad_rows" and t.get(k) else t.get(k)) for k in keep if k in t}
                                for n, t in (v.get("tensors") or {}).items()}}
                for L, v in ls.items()} for w, ls in scans.items()}
    changed = []
    a, b = scans.get("after_postprocess") or {}, scans.get("after_forward_0") or {}
    for L in a:
        for n, t in ((a[L] or {}).get("tensors") or {}).items():
            u = (((b.get(L) or {}).get("tensors")) or {}).get(n) or {}
            for k in ("sum", "wsum", "nonfinite", "min", "max"):
                if u and k in t and t.get(k) != u.get(k):
                    changed.append({"layer": L, "tensor": n, "field": k, "before": t.get(k), "after": u.get(k)})
    nf = [r for r in recs if r.get("ev") == "nan_forward"]
    by_mode = {}
    for r in nf:
        m = r.get("ple_mode") or "orig"
        d = by_mode.setdefault(m, {"forwards": 0, "with_nonfinite": 0, "first_module": None, "logits_nonfinite": 0})
        d["forwards"] += 1
        bad = bool(r.get("first_nonfinite")) or bool((r.get("logits") or {}).get("nonfinite"))
        d["with_nonfinite"] += bad
        d["logits_nonfinite"] += bool((r.get("logits") or {}).get("nonfinite"))
        if bad and d["first_module"] is None:
            d["first_module"] = ((r.get("first_nonfinite") or [{}])[0]).get("module") or "logits"
    first_moe = next((r.get("moe_in") for r in nf if r.get("moe_in")), None)
    src = next((v.get("src") for ls in scans.values() for v in ls.values() if isinstance(v, dict) and v.get("src")), None)
    return {"runtime_src": src, "scans": comp, "changed_after_forward0": changed[:40], "by_ple_mode": by_mode, "first_moe_in": first_moe,
            "events": [{k: r.get(k) for k in ("ev", "mode", "module", "prefix", "qm", "error") if r.get(k) is not None}
                       for r in recs if str(r.get("ev", "")).startswith(("rc1_", "ple_mode"))][:30]}


def _e13_gptq_dequant(qweight, qzeros, scales, group=128):
    """GPTQ (auto_gptq v1 packing, AutoRound 'auto_round:auto_gptq'): qweight [K/8, N] int32 (8 nibbles along K),
    qzeros [K/g, N/8] int32 (8 nibbles along N, stored zero - 1), scales [K/g, N] -> W [K, N] float32."""
    import torch
    sh = torch.arange(0, 32, 4, dtype=torch.int32)
    q = ((qweight.unsqueeze(1) >> sh.view(1, 8, 1)) & 0xF).reshape(-1, qweight.shape[1])         # [K, N]
    zs = ((qzeros.unsqueeze(2) >> sh.view(1, 1, 8)) & 0xF).reshape(qzeros.shape[0], -1)          # [K/g, N] stored
    z = zs if bool((zs == 8).all()) else zs + 1     # v2 stores the zero (8 for sym int4); v1 stores zero - 1 (7)
    g = torch.arange(q.shape[0]) // group
    return (q.float() - z[g].float()) * scales.float()[g], int(torch.unique(zs).numel()), int(zs.flatten()[0])


def e13_rc1_reference(tdir, model_dir, deadline):
    """RC1 (GPT audit 3, test B; RC1 audit H1/H2): for every saved MoE fixture (layers 1 and 2, first forwards, per rank)
    recompute the MoE output on the CPU from the checkpoint (GPTQ dequant + PyTorch SiLU-MLP; a fused shared-expert slot
    from its bf16 weights) and compare with the Marlin output of the same tokens.
    EP semantics from SGLang 0.5.21 (token_dispatcher/standard.py): FusedMoE receives GLOBAL top-k ids; the dispatcher maps
    them with local_expert_mapping (global -> local index, -1 = not on this rank). The ownership comes only from the
    captured map (never inferred from the id range); a fixture without it is not compared. The module output is the
    rank-local sum unless reduce_results is set (then all-reduced = the global sum).
    Also scans the checkpoint's scales of layers 1-2. Layer 1 (finite Marlin output) validates the reference itself.
    Runs after all GPU arms; every loop checks `deadline` (a truncated result says so)."""
    import torch
    from safetensors import safe_open
    t0 = time.perf_counter()
    left = lambda: deadline - time.perf_counter()
    out = {"fixtures": [], "ckpt_scales": {}}
    try:
        wm = json.loads((Path(model_dir) / "model.safetensors.index.json").read_text())["weight_map"]
    except Exception as e:
        return {"error": f"index: {type(e).__name__}: {e}"[:300]}
    handles = {}

    def get(name):
        f = wm.get(name)
        if f is None:
            return None
        if f not in handles:
            handles[f] = safe_open(str(Path(model_dir) / f), framework="pt")
        return handles[f].get_tensor(name)
    pre = next((p for p in ("model.language_model.layers.", "model.layers.", "language_model.model.layers.")
                if any(k.startswith(p) for k in wm)), "model.language_model.layers.")
    for L in (1, 2):                                                 # checkpoint scales: every expert, every projection
        bad, n, mx = [], 0, 0.0
        names = [k for k in wm if k.startswith(f"{pre}{L}.mlp.experts.") and k.endswith(".scales")]
        for k in names:
            if left() < 60:
                out["truncated"] = f"deadline during the checkpoint scale scan of layer {L}"; break
            s = get(k).float(); n += 1
            if not torch.isfinite(s).all():
                bad.append(k)
            else:
                mx = max(mx, float(s.abs().max()))
        out["ckpt_scales"][str(L)] = {"tensors": n, "of": len(names), "nonfinite": bad[:10], "nonfinite_n": len(bad), "absmax": mx}
    cache = __import__("collections").OrderedDict()                  # bounded: 24 experts x ~18.75 MiB fp32

    def _put(key, val):
        cache[key] = val
        while len(cache) > 24:
            cache.popitem(last=False)

    class _Late(Exception):
        pass

    def expert(L, e):
        if (L, e) not in cache:
            if left() < 30:
                raise _Late(f"deadline before loading expert {e} of layer {L}")
            W = {}
            for p in ("gate_proj", "up_proj", "down_proj"):
                b = f"{pre}{L}.mlp.experts.{e}.{p}."
                qw, qz, sc = get(b + "qweight"), get(b + "qzeros"), get(b + "scales")
                if qw is None:
                    return None
                W[p], W[p + "_zero_unique"], W[p + "_zero0"] = _e13_gptq_dequant(qw, qz, sc)
            _put((L, e), W)
            zero_seen.update({k: v for k, v in W.items() if "zero" in k})
        cache.move_to_end((L, e))
        return cache[(L, e)]

    def shared(L):                                                   # a fused shared-expert slot: bf16 weights [out, in]
        if (L, "shared") not in cache:
            W = {}
            for p in ("gate_proj", "up_proj", "down_proj"):
                w = get(f"{pre}{L}.mlp.shared_expert.{p}.weight")
                if w is None:
                    return None
                W[p] = w.float().t()
            _put((L, "shared"), W)
        cache.move_to_end((L, "shared"))
        return cache[(L, "shared")]
    rel = lambda a, b: float((a - b).norm() / max(float(b.norm()), 1e-9))
    zero_seen = {}
    for fp in sorted(Path(tdir).glob("fixture_L*_fw*_pid*.pt")):
        if left() < 30:
            out["truncated"] = out.get("truncated") or "deadline before all fixtures"; break
        r = {"file": fp.name}
        try:
            fx = torch.load(fp, map_location="cpu", weights_only=False)
            L = int(re.search(r"layers\.(\d+)\.", fx["name"]).group(1))
            meta = fx.get("meta") or {}
            x = fx["hidden"].float()
            T = x.shape[0]
            ids = next((t for t in fx["ints"] if t is not None and t.dim() == 2 and t.shape[0] == T), None)
            wts = next((t for t in fx["floats"] if t is not None and t.dim() == 2 and ids is not None and t.shape == ids.shape), None)
            em = fx.get("expert_map")
            r.update(layer=L, fw=fx.get("fw"), rank=fx.get("rank"), meta=meta, tokens=T, hidden_from=fx.get("hidden_from"),
                     ids=None if ids is None else ids.tolist(), weights=None if wts is None else [[round(float(v), 4) for v in row] for row in wts])
            if ids is None or wts is None:
                r["error"] = f"no top-k ids/weights among ints {fx.get('int_shapes')} floats {fx.get('float_shapes')}"
                out["fixtures"].append(r); continue
            nfs = int(meta.get("num_fused_shared_experts") or 0)
            ne = int(meta.get("num_experts") or 512)
            ner = ne - nfs                                           # routed experts (checkpoint indices 0 .. ner-1)
            if em is None:
                if int(meta.get("moe_ep_size") or 1) > 1:
                    r["error"] = "missing expert_map; EP mapping unverified (not compared)"
                    out["fixtures"].append(r); continue
                em = torch.arange(ne, dtype=torch.int32)             # no EP: every expert local, identity map
            em = em.to(torch.int64).flatten()
            if int(ids.max()) >= em.numel() or int(ids.min()) < -1:
                r["error"] = f"top-k ids outside the expert map: min {int(ids.min())} max {int(ids.max())} map {em.numel()}"
                out["fixtures"].append(r); continue
            # audit extra A: the map's ownership must be one contiguous block of routed experts numbered 0.. in order
            owned = [g for g in range(min(ner, em.numel())) if int(em[g]) >= 0]
            r["mapping"] = {"owned_routed": len(owned), "owned_first": owned[:1], "owned_last": owned[-1:],
                            "contiguous_in_order": bool(owned) and owned == list(range(owned[0], owned[0] + len(owned)))
                                                   and [int(em[g]) for g in owned] == list(range(len(owned))),
                            "selected_owned": int(sum(1 for v in ids.flatten().tolist() if v >= 0 and int(em[v]) >= 0)),
                            "selected_total": int((ids >= 0).sum())}
            ref_glob = torch.zeros(T, x.shape[1]); ref_loc = torch.zeros(T, x.shape[1])
            exp_bad = []
            for t in range(T):
                for k in range(ids.shape[1]):
                    g = int(ids[t, k])
                    if g < 0:
                        continue
                    W = expert(L, g) if g < ner else shared(L)
                    if W is None:
                        r.setdefault("missing_experts", []).append(g); continue
                    xt = x[t:t + 1]
                    h = torch.nn.functional.silu(xt @ W["gate_proj"]) * (xt @ W["up_proj"])
                    y = (h @ W["down_proj"])[0] * float(wts[t, k])
                    if not torch.isfinite(y).all():
                        exp_bad.append(g)
                    ref_glob[t] += y
                    if int(em[g]) >= 0:
                        ref_loc[t] += y
            mo = fx["out"].float() if fx.get("out") is not None else None
            red = bool(meta.get("reduce_results"))
            ref = ref_glob if red else ref_loc
            r.update(compare_to="global (reduce_results)" if red else "rank-local (reduce_results False)",
                     ref_finite=bool(torch.isfinite(ref).all()), ref_experts_nonfinite=sorted(set(exp_bad))[:10],
                     ref_absmax=float(ref.abs().max()), marlin_finite=None if mo is None else bool(torch.isfinite(mo).all()),
                     zero_convention=dict(zero_seen))
            if mo is not None and mo.shape == ref.shape and bool(torch.isfinite(mo).all()):
                r.update(rel_err=round(rel(mo, ref), 5), rel_err_other_sum=round(rel(mo, ref_loc if red else ref_glob), 5))
            elif mo is not None and mo.shape == ref.shape:
                r["marlin_nonfinite_tokens"] = int((~torch.isfinite(mo)).any(1).sum())
            elif mo is not None:
                r["error"] = f"Marlin output shape {list(mo.shape)} != reference {list(ref.shape)}"
        except _Late as e:
            r["error"] = str(e); out["truncated"] = str(e)
            out["fixtures"].append(r); break
        except Exception as e:
            r["error"] = f"{type(e).__name__}: {e}"[:400]
        out["fixtures"].append(r)
    # verdict per the GPT audit table, only when layer 1 validates the reference (finite Marlin, rel err < 5 %)
    by = {}
    for r in out["fixtures"]:
        by.setdefault(r.get("layer"), []).append(r)
    l1 = [r for r in by.get(1, []) if r.get("rel_err") is not None]
    l2 = [r for r in by.get(2, []) if not r.get("error")]
    out["reference_validated_on_layer1"] = bool(l1) and max(r["rel_err"] for r in l1) < 0.05
    if not out["reference_validated_on_layer1"]:
        out["verdict"] = "INCONCLUSIVE: reference not validated on layer 1"
    elif not l2:
        out["verdict"] = "INCONCLUSIVE: no usable layer-2 fixture"
    elif any(r.get("marlin_finite") is False for r in l2) and all(r.get("ref_finite") for r in l2):
        out["verdict"] = "MARLIN PATH FAULT: Marlin NaN on layer 2, the checkpoint reference is finite"
    elif any(r.get("marlin_finite") is False for r in l2):
        out["verdict"] = "WEIGHTS / ROUTING: Marlin and the reference are both non-finite on layer 2"
    elif all(r.get("rel_err") is not None and r["rel_err"] < 0.05 for r in l2):
        out["verdict"] = "MOE OK on these tokens: Marlin = reference on layer 2 (look at stream / integration)"
    else:
        out["verdict"] = "MISMATCH: layer-2 Marlin finite but differs from the reference"
    out["sec"] = round(time.perf_counter() - t0, 1)
    return out


# ---- RC1 arm 2: Flash-Next on vLLM 0.32.0 (native qwen4_exp). Install = the E18 VMS installer (Kaggle-proven).
_E13_VLLM_INST = None
E13_VLLM_HOOK_SRC = __import__("zlib").decompress(_b64.b64decode("eNqVWG1v2zYQ/u5fcVM+TGpdzUmLYfDiDlmTthuSJkiyAUNXCLRE2Wok0iGpJF7R/76HpGRbkeNuBhzbvLeHx7vnqOzRyf5LunyzT3enp2c0l/KGQl0YntbayKr4h5MUZObcyzVXd1x9r+nir+v35x8ujq7fQ14uf6ZCcGXovjBzWRvrNPkTBsn15dGbk+T4t8soHuzR1btTJmYwP2OqLASdyRNSfMHSGzo/P+OZjfXq4fQVTXnKao3YZUbTOs+50nTPFadccadWLmm6dLhmTE3ZjFMqy5KnRioKT8+Pji8uz389GdPzUfxyRO+KXxF9wZWLWLIlV1HsdwT5QYNBYxNUyYyXSSlZxlVcm6LU8ULJlGud3PNiNjc6YbnhyqkUYkahFJxKKRckkRpiZWl91CXX0dgCROB7xRY2eMVuuCYOtaUDclszYajiyFlGqhY0S+NmG2FELgwVRpO8F7QTxJCYyABipl1K3l38gagVryQCtbv2oGhR1poY6bqqmFrGdA70v1+dfzilvLBiaDehxnTYP8bXP9wZxVKeHC6K7HX8WeMo4kFRLSROX+r2m17qwSA5pgnWYi7uCiVFPOMmDPoug2gwGGQ8J+xmFiqeRuMB4dW4siE2f5ui4u43NOE/K5AsfB2SmShZiyy0CrH9E0ZDeok3kE4AA+HxLYwiZ23U0oexL0jt2WSF0mFyPCT+UGiTyJvJtap5tFKz1U1ywUUIgwUz8/izLISzyIMmL182In31+QmAIWAB3IpU2uOaBLXJX/wU4Iw15WsY9pXH9wrtF1rLOKurhfa7Q4JYXZqJNiqi5xT8LQIPjD+kfIGOcx+FFGt3C6Z1m1qUpUzDJrOdvbdplSqdrxYVN7USfjFO64zFvpq8H2Y4dkdFvqlQoCbvWFGyaYnMEy/RvqNvIGzCjFqUyGk6D1GpDVCpihnOGAu7m9ApawPVL4EIxjRCwh1T2O9fB05sA+x0ErrWH9Izhvezm2gNc88RJKuzwoxdM7suc937ouneN6dHV1eWC1LfQ2mJ5OPcBXekIEAt2jBIx2T5otP5cvoZHQ92AKfxjaB6zhTIDjxXMbH0tKV9q9v4LasYxyp2xWmAImXlaIOpWV1xYYakJQlWQcvapjhDjWwxzw3OKH5cDrN0teItkVqUdAVOE5SDZMWQqhVfxlYnSxreQ+GvW8aBzGD+8dNq0dnDYpeLblfcVvCAtmLGqLDC6boEJj6B6KwPSGrUsUhLi9ksFzy8rboi1C38FdpZocYoqCQPSEhj0cAwThKLJkniUmLqoJqlcg4hbjbUhecCSmEKUfNuNwODd2h5Cg4dB+4sw6AHNm+xjmnXaw/w5tySR9ZUKUNbz2HritEWga6n7jtvzr8WRtbpnGe93eQb+cYGkPHdoLcdwSZ0m8HWXw5vCcccmKWwe8vAFNGOfHYktotnoeYlnLjKtf2Khk3ySe7a9maLq+kIu2kpsCdVVphv8em89Z3t73K2OcH7pge7TLX5CPL6RM8ntL9V6BnNKVTsIQTJAcsLeO37Qupbd4cTemmz3+jSa7j/kQ4P6WDUz5R9uTH8JeB3IM/gzp1R4NsSC44MXBWDDfxFCuJf3Ii7raDgOs7mMlp1EURTjp639tPRcPB0CQe+qBb3rLS6+8N2BSjw++DrltPzI0R1BLO4qS/k247wjrAz/Vb5/V+1PuvjaHghtn9xCYGfrs7jIWhHP+/jeJz8xPpNuFJS9c4Bv71gbC+wKuTRx/HBaPRpI0tddzavCRpfGevL1SHW24LEUrOJpCEJCD+mq2N0tJ1ucOBmoF5Sm3OxE/zxYF3PgUIg9iPDLZCR0CcAg7jbkQFBU/Lt8E+mS7Nab5oHkF20b14pUDnfvnI8wmpvLzzrnJON0ybQxh6gM5PjzgXXf5TFNGbTdLugYuncPmAtPXjP58nbQuAhJezYx2co5AtcTb1wgwyvjy7fnVxjW8FdWVaxf87hD3jQwyUu7j/2BGvitaSLs8oSveBpQ5S+AO0lGDdv3DS4mbgJMH48AqwifTchaxZ7EOOnuth66MhsQEDekol4vcl4jW2NqjeKnKuNaWR/x37Dq/n6FC6r25HZsnbJA7gNR7FdagqyP7c2hKH/2DKrVp5blZ7GVv5y1bi6Pm+1+48M9ERdrzmoxzmvupzTntuWnCBbmxl6KtP+Or/EQxmKOXEPWrg+4/8Ldua1ZR9Fg38BXWTmnw==")).decode()   # gc after each MoE repack


def e13_vllm_install(deadline=None):
    global _E13_VLLM_INST
    if _E13_VLLM_INST is not None:
        return _E13_VLLM_INST
    if os.environ.get("E13_VLLM_FAKE"):                                 # local harness: a fake OpenAI server
        _E13_VLLM_INST = {"fake": True, "launch": [sys.executable, os.environ["E13_VLLM_FAKE"]], "env": dict(os.environ), "sec": 0}
        return _E13_VLLM_INST
    _whd = Path("/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse")
    wh = sorted(_whd.rglob("install_wheelhouse.py")) if _whd.is_dir() else []
    if not wh:
        raise E11Fail("vLLM 0.32 wheelhouse not attached (foysalemonshanto/vllm-0-32-0-cu13-wheelhouse)")
    t = time.perf_counter()
    r = subprocess.run([sys.executable, str(wh[0]), "/tmp/vllm-venv"], capture_output=True, text=True,
                       timeout=max(60, min(2400, HARD_STOP - time.perf_counter() - 150,
                                           (deadline or HARD_STOP) - time.perf_counter() - 60)))
    if r.returncode:
        raise E11Fail("vLLM 0.32 install failed: " + ((r.stdout or "") + (r.stderr or ""))[-2000:])
    kv = dict(l.split("=", 1) for l in (r.stdout or "").splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "LD_PRELOAD")}
    env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"/tmp/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", VLLM_LOGGING_LEVEL="INFO",
               VLLM_CACHE_ROOT="/tmp/vllm-cache", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", OMP_NUM_THREADS="8")
    _E13_VLLM_INST = {"launch": ["/tmp/vllm-venv/bin/vllm", "serve"], "env": env, "sec": round(time.perf_counter() - t),
                      "venv_python": kv.get("VENV_PYTHON")}
    return _E13_VLLM_INST


def e13_vllm_flags():
    """vLLM 0.32.0 serve flags for the first Flash-Next smoke (GPT audit 2: target only, eager, short context, low
    concurrency): TP4 + expert parallel (= SGLang's EP4 layout, full experts per rank -> Marlin), PLE table in pinned host
    memory, language model only (no vision profiling), bf16 KV (the default), no prefix caching (state-free smoke)."""
    return [str(E13_INTEL_DIR), "--served-model-name", "flashnext", "--host", "127.0.0.1", "--port", str(E13_VLLM_PORT),
            "--tensor-parallel-size", "4", "--enable-expert-parallel", "--enforce-eager",
            "--max-model-len", "8192", "--max-num-seqs", "4", "--max-num-batched-tokens", "4096",
            "--gpu-memory-utilization", "0.93", "--disable-custom-all-reduce", "--language-model-only",
            "--engram-config", json.dumps({"cpu_offload": True}), "--distributed-executor-backend", "mp",
            "--load-format", "safetensors", "--dtype", "bfloat16", "--no-enable-prefix-caching", "--seed", "0"]


def _e13_vpost(path, body, deadline, cap):
    left = min(cap, deadline - time.perf_counter())
    if left < 20:
        raise TimeoutError(f"arm deadline: {round(left)} s left")
    req = _urlreq.Request(f"http://127.0.0.1:{E13_VLLM_PORT}" + path, data=json.dumps(body).encode(),
                          headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=left) as r:
        return json.loads(r.read())


def e13_vllm_smoke(deadline):
    out = {}
    try:
        t = time.perf_counter()
        c = _e13_vpost("/v1/completions", {"model": "flashnext", "prompt": "1, 2, 3, 4, 5, 6, 7, 8,", "max_tokens": 32,
                                           "temperature": 0, "logprobs": 1}, deadline, 600)
        ch = (c.get("choices") or [{}])[0]
        text = ch.get("text") or ""
        lp = (ch.get("logprobs") or {}).get("token_logprobs") or []
        lpv = [float(v) for v in lp if v is not None]
        out.update(count_text=text[:80], count_ok=text.strip().startswith("9, 10, 11"), garbage=_e13_garbage(text),
                   count_logprobs=[None if v is None else round(float(v), 4) for v in lp[:8]],
                   logprobs_ok=bool(lpv) and all(math.isfinite(v) for v in lpv), count_sec=round(time.perf_counter() - t, 1))
        t = time.perf_counter()
        m = _e13_vpost("/v1/chat/completions", {"model": "flashnext", "max_tokens": 64, "temperature": 0,
                                                "chat_template_kwargs": {"enable_thinking": False}, "messages": [
            {"role": "user", "content": "What is 2 + 2? Answer with the number only."}]}, deadline, 600)
        msg = (m.get("choices") or [{}])[0].get("message") or {}
        content = msg.get("content") or ""
        answer = content.split("</think>")[-1].strip()           # the reply after any thinking, nothing from the reasoning
        out.update(chat_content=content[:160], chat_answer=answer[:40], chat_ok=re.fullmatch(r"\**4\**\s*\.?", answer) is not None,
                   chat_usage=m.get("usage"), chat_finish=(m.get("choices") or [{}])[0].get("finish_reason"),
                   chat_sec=round(time.perf_counter() - t, 1))
        res = [None] * 4

        def one(i):
            try:
                j = _e13_vpost("/v1/completions", {"model": "flashnext", "prompt": f"Write a long story about grid puzzle {i}.",
                                                   "max_tokens": 256, "temperature": 0.7, "ignore_eos": True}, deadline, 900)
                res[i] = (j.get("usage") or {}).get("completion_tokens")
            except Exception as e:
                res[i] = f"{type(e).__name__}: {e}"[:160]
        t = time.perf_counter()
        ths = [threading.Thread(target=one, args=(i,), daemon=True) for i in range(4)]
        for th in ths:
            th.start()
        for th in ths:
            th.join(max(1.0, deadline - time.perf_counter()))
        sec = time.perf_counter() - t
        toks = sum(x for x in res if isinstance(x, int))
        out.update(streams4=res, streams4_sec=round(sec, 1), streams4_tok_s=round(toks / max(sec, 1e-6), 1),
                   streams4_ok=len(res) == 4 and all(type(n) is int and n == 256 for n in res))
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"[:400]
    # RC1 audits: correctness = count + finite logprobs + the exact answer; throughput only counts with all 4 streams
    out["smoke_correct"] = (bool(out.get("count_ok")) and not out.get("garbage") and bool(out.get("logprobs_ok"))
                            and bool(out.get("chat_ok")) and not out.get("error"))
    out["throughput_valid"] = out["smoke_correct"] and bool(out.get("streams4_ok"))
    out["ok"] = out["throughput_valid"]
    return out


_E13_VLLM_LOG_KEYS = ("PLE", "Engram", "engram", "ngram", "quantiz", "INC", "Marlin", "marlin", "MoE", "KV cache",
                      "Maximum concurrency", "QSA", "FlashAttention", "fallback", "Traceback", "Error", "ERROR", "out of memory")


def e13_vllm_arm(tag, cfg, budget):
    """RC1 arm 2. Never raises for an engine outcome (that is the measurement)."""
    global E11_SERVER
    rec = {"tag": tag, "engine": "vllm 0.32.0", "model": str(E13_INTEL_DIR), "budget_sec": round(budget)}
    rec["host_ram"] = e13_host_ram_guard()
    if not rec["host_ram"].get("ok", False):                           # RC1 audits: fail closed like the SGLang arm
        rec["outcome"] = "skipped: host RAM guard"; return rec
    rec["gpu_free_before"] = _e11_gpu_free(180)
    if rec["gpu_free_before"].get("free") is not True:
        rec["outcome"] = "skipped: GPU free status not verified"; return rec
    t_deadline = min(time.perf_counter() + budget, HARD_STOP - 5 * 60)
    try:
        inst = e13_vllm_install(t_deadline)
    except Exception as e:
        rec["outcome"] = f"install error: {type(e).__name__}: {e}"[:1500]; return rec
    if time.perf_counter() > t_deadline - 12 * 60:                     # no launch that cannot reach ready + smoke
        rec["outcome"] = f"skipped: {round(t_deadline - time.perf_counter())} s left after the install"; return rec
    rec["install_sec"] = inst.get("sec")
    rec["flags"] = e13_vllm_flags()
    _vh = Path(os.environ.get("E13_VLLM_HOOK_DIR", "/tmp/e13_vllm_hook")); _vh.mkdir(parents=True, exist_ok=True)
    (_vh / "sitecustomize.py").write_text(E13_VLLM_HOOK_SRC, encoding="utf-8")
    vtd = OUT / "e13_memtrace" / tag
    venv_env = dict(inst["env"], PYTHONPATH=str(_vh), E13_VLLM_TRACE_DIR=str(vtd))
    E11_SERVER = E11Server(inst["launch"] + rec["flags"], venv_env, f"e13_{tag}")
    E11_SERVER.log_path = OUT / f"vllm_server_e13_{tag}.log"
    E11_SERVER.start()
    print(f"E13 RC1 {tag}: vLLM launch", flush=True)
    last, last_t = -1, time.perf_counter()
    while True:
        if not E11_SERVER.alive():
            rec["outcome"] = f"exited rc={E11_SERVER.proc.returncode}"; break
        try:
            _urlreq.urlopen(f"http://127.0.0.1:{E13_VLLM_PORT}/health", timeout=10).read(); rec["outcome"] = "ready"; break
        except Exception:
            pass
        sz = E11_SERVER.log_path.stat().st_size if E11_SERVER.log_path.exists() else 0
        if sz != last:
            last, last_t = sz, time.perf_counter()
        if time.perf_counter() - last_t > 10 * 60:
            rec["outcome"] = "stall (10 min log silence)"; break
        if time.perf_counter() > t_deadline - 4 * 60:
            rec["outcome"] = "timeout"; break
        time.sleep(2 if E11_HARNESS else 5)
    rec["sec"] = round(time.perf_counter() - E11_SERVER.t0, 1)
    if rec["outcome"] == "ready":
        rec["smoke"] = e13_vllm_smoke(t_deadline)
    log = E11_SERVER.log_text()
    lines = log.splitlines()
    rec["log_lines"] = [l[:300] for l in lines if any(k in l for k in _E13_VLLM_LOG_KEYS)][:80]
    rec["kv_tokens"] = [int(x.replace(",", "")) for x in re.findall(r"GPU KV cache size: ([\d,]+) tokens", log)][:4]
    rec["max_concurrency"] = re.findall(r"Maximum concurrency for [^\n]{0,80}", log)[:2]
    rec["oom_lines"] = [l[:300] for l in lines if "out of memory" in l.lower()][:6]
    rec["log_tail"] = log[-3000:] if rec["outcome"] != "ready" or not (rec.get("smoke") or {}).get("ok") else ""
    E11_SERVER.kill()
    if not E11_HARNESS:
        subprocess.run(["pkill", "-9", "-f", "vllm"], capture_output=True)
    rec["gpu_free_after"] = _e11_gpu_free(180)
    vrecs = []
    for _vf in sorted(vtd.glob("vtrace_*.jsonl")):
        for _l in _vf.read_text(encoding="utf-8", errors="replace").splitlines():
            try:
                vrecs.append(json.loads(_l))
            except ValueError:
                pass
    rec["vllm_gc"] = {"patched": sum(r.get("ev") == "vpatched" for r in vrecs),
                      "errors": [r for r in vrecs if "error" in str(r.get("ev"))][:4],
                      "pwal_end": [{k: r.get(k) for k in ("pid", "alloc", "moe_modules", "freed_bytes")} for r in vrecs if r.get("ev") == "vpwal_end"][:4],
                      "samples": [r for r in vrecs if r.get("ev") == "vgc"][:6]}
    print(f"E13 RC1 {tag}:", json.dumps({k: rec.get(k) for k in ("outcome", "sec", "kv_tokens", "smoke", "oom_lines")},
                                       default=str)[:2000], flush=True)
    return rec


def e13_probe_arm(tag, model_dir, vpy, hookdir, budget):
    """One launch with the rev 6b flags on `model_dir`, the memory trace on; until ready / exit / stall / budget; then a
    smoke if ready; then kill and wait for free GPUs. Never raises for an engine outcome (that is the measurement)."""
    global SGL_MODEL_PATH, E11_SERVER
    rec = {"tag": tag, "model": str(model_dir), "budget_sec": round(budget)}
    SGL_MODEL_PATH = Path(model_dir)
    rec["host_ram"] = e13_host_ram_guard()
    if not rec["host_ram"]["ok"]:
        rec["outcome"] = "skipped: host RAM guard"; return rec
    rec["gpu_free_before"] = _e11_gpu_free(180)
    if rec["gpu_free_before"].get("free") is not True:               # fix list F17 + RC1 audit: fail closed
        rec["outcome"] = "skipped: GPU free status not verified"; return rec
    t_deadline = min(time.perf_counter() + budget, HARD_STOP - 5 * 60)  # F10: smoke + long test inside the arm too
    tdir = OUT / "e13_memtrace" / tag
    _e13_set_ple_mode("orig")                                          # RC1: every arm starts on SGLang's own PLE path
    env = dict(_E13_ENV); env["PYTHONPATH"] = str(hookdir); env["E13_MEMTRACE_DIR"] = str(tdir)
    cfg = (globals().get("E13_ARM_CFG") or {}).get(tag) or {}          # NANPROBE: per-arm env / flag changes
    env.update(cfg.get("env") or {})
    fl = e13_flags()
    for k, v in (cfg.get("set") or {}).items():
        i = fl.index(k)
        fl[i + 1] = v
    rec["flags"] = fl + list(cfg.get("add") or [])
    rec["arm_cfg"] = cfg
    launch = [str(vpy), "-m", "sglang.launch_server"] if not E11_HARNESS else [sys.executable, os.environ["E11_FAKE_SERVER"]]
    E11_SERVER = E11Server(launch + rec["flags"], env, f"e13_{tag}")
    E11_SERVER.start()
    print(f"E13 LOADPROBE {tag}: launch on {model_dir}", flush=True)
    t_end, last, last_t = t_deadline - (cfg.get("reserve_after_ready_sec") or 0), -1, time.perf_counter()
    while True:
        if not E11_SERVER.alive():
            rec["outcome"] = f"exited rc={E11_SERVER.proc.returncode}"; break
        try:
            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read(); rec["outcome"] = "ready"; break
        except Exception:
            pass
        sz = E11_SERVER.log_path.stat().st_size if E11_SERVER.log_path.exists() else 0
        if sz != last:
            last, last_t = sz, time.perf_counter()
        if time.perf_counter() - last_t > 10 * 60:
            rec["outcome"] = "stall (10 min log silence)"; break
        if time.perf_counter() > t_end:
            rec["outcome"] = "timeout"; break
        time.sleep(2 if E11_HARNESS else 5)
    rec["sec"] = round(time.perf_counter() - E11_SERVER.t0, 1)
    log = E11_SERVER.log_text()
    rec["caps"] = _e13_caps(log)
    rec["oom_lines"] = [l[:300] for l in log.splitlines() if "out of memory" in l.lower()][:6]
    rec["load_weight_end_lines"] = [l[:300] for l in log.splitlines() if "Load weight end" in l][:4]
    if rec["outcome"] == "ready" and cfg.get("ple_modes"):             # RC1: one count per PLE mode, same server
        rec["ple_runs"] = e13_ple_mode_runs(t_deadline, cfg["ple_modes"])
        rec["smoke"] = (rec["ple_runs"] or [{}])[0]
    elif rec["outcome"] == "ready" and cfg:                            # NANPROBE arms (rev 2)
        rec["smoke"] = e13_probe_smoke(t_deadline, cfg)
        if cfg.get("rerun_count") and E11_SERVER.alive():              # F14: drain past the hook window, then count again
            rec["count_after_hooks"] = e13_probe_smoke(t_deadline, {**cfg, "chat_max_tokens": 0, "drain_tokens": 160})
        if cfg.get("capacity_gate"):                                   # F8: the KV figure must belong to 4 streams
            cap = rec["caps"].get("capped_running")
            rec["capacity_gate"] = "OK" if cap in (None, SGL_SERVER_MAX_RUNNING) else f"CAPACITY_GATE_FAILED (capped to {cap})"
        if cfg.get("long_test") and E11_SERVER.alive():                # F9: runtime memory headroom with long prefills
            rec["long_test"] = e13_long_test(t_deadline, cfg["long_test"])
        rec["caps_after"] = _e13_caps(E11_SERVER.log_text())
    elif rec["outcome"] == "ready":
        try:
            rec["smoke"] = e13_smoke()
        except Exception as e:
            rec["smoke"] = {"ok": False, "error": f"{type(e).__name__}: {e}"[:400]}
    E11_SERVER.kill()
    rec["gpu_free_after"] = _e11_gpu_free(180)
    if (cfg.get("env") or {}).get("E13_RC1_FIXTURE") == "1":            # RC1 test B: CPU reference, after ALL GPU arms
        E13_RC1_PENDING.append((tag, tdir, model_dir))
        rec["rc1_reference"] = {"pending": "runs after all GPU arms (RC1 audit H1)"}
    rec["memtrace"] = e13_memtrace_summary(tdir)
    print(f"E13 LOADPROBE {tag}:", json.dumps({k: rec.get(k) for k in ("outcome", "sec", "caps", "oom_lines", "smoke")},
                                             default=str)[:1500], flush=True)
    print(f"E13 LOADPROBE {tag} memtrace:", json.dumps(rec["memtrace"], default=str)[:4000], flush=True)
    return rec


_E13_ENV = None
_t_engine = time.perf_counter()
E13_RC1_PENDING = []          # RC1: (arm tag, trace dir, model dir) whose CPU reference runs after all GPU arms
E13_PROBE = {"arms": [], "budget_hard_stop_in_sec": round(HARD_STOP - time.perf_counter())}
ENGINE_INFO["e13"]["probe"] = E13_PROBE
if not E10_SANDBOX_SELFTEST.get("ok", False):
    ENGINE_INFO["e13"]["stage"] = "skipped: sandbox self-test failed"
else:
    try:
        _info = ENGINE_INFO["e13"]
        _info["stage"] = "install"
        _vpy, _E13_ENV, _info["install"] = e11_install()
        _E13_ENV.update({"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
                         "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false", "SGLANG_MM_PREPROCESS_DEVICE": "cpu",
                         "OMP_NUM_THREADS": "8", "TORCH_CUDA_ARCH_LIST": "8.9"})                      # = e13_bring_up
        _info["stage"] = "fa2_patch3"; _info["extras"] = e13_extras(_vpy)
        _info["stage"] = "topk_test"; _info["topk"] = e13_topk_test(_vpy)
        print("E13 top-k test:", json.dumps(_info["topk"], default=str)[:1500], flush=True)
        _hook = Path(os.environ.get("E13_HOOK_DIR", "/tmp/e13_memtrace_hook")); _hook.mkdir(parents=True, exist_ok=True)
        (_hook / "sitecustomize.py").write_text(E13_MEMTRACE_SRC, encoding="utf-8")
        _arms = list(E13_PROBE_ARMS)
        while _arms:
            _tag = _arms.pop(0)
            _room = HARD_STOP - time.perf_counter() - 5 * 60
            if _room < 8 * 60:
                E13_PROBE["arms"].append({"tag": _tag, "outcome": f"skipped: {round(_room)} s left"}); continue
            _info["stage"] = f"probe {_tag}"
            _acfg = (globals().get("E13_ARM_CFG") or {}).get(_tag) or {}
            if _room < float(_acfg.get("min_room_sec") or 0):                 # RC1: the optional third arm
                E13_PROBE["arms"].append({"tag": _tag, "outcome": f"skipped: {round(_room)} s left (< {_acfg['min_room_sec']})"}); continue
            try:
                if _acfg.get("engine") == "vllm":                                  # RC1 arm 2: Flash-Next on vLLM 0.32
                    E13_PROBE["arms"].append(e13_vllm_arm(_tag, _acfg, min(float(_acfg.get("budget_sec") or 30 * 60), _room)))
                    (OUT / "e13_loadprobe.json").write_text(json.dumps(E13_PROBE, indent=1, default=str), encoding="utf-8")
                    continue
                if _tag == "bf16ple" or _acfg.get("model") == "intel":
                    _md = E13_INTEL_DIR
                else:
                    _md, E13_PROBE["fp8ple_merge"] = e13_fp8ple_dir()
                _r = e13_probe_arm(_tag, _md, _vpy, _hook, min(float(_acfg.get("budget_sec") or 25 * 60), _room))
                E13_PROBE["arms"].append(_r)
                # NANPROBE rev 2 (F13): an OOM before ready on the NaN arm -> one retry with gc after every module; the KV
                # arm is dropped then (it would OOM the same way and Q1 matters more)
                if _acfg.get("oom_retry") and _r.get("outcome") != "ready" and (_r.get("oom_lines") or _r.get("caps", {}).get("oom")):
                    _arms = [_acfg["oom_retry"]] + [a for a in _arms if a not in ("kv_lean",)]
                    E13_PROBE["oom_retry"] = {"after": _tag, "next": _acfg["oom_retry"], "dropped": "kv_lean"}
            except Exception as _ae:
                E13_PROBE["arms"].append({"tag": _tag, "outcome": f"error: {type(_ae).__name__}: {_ae}"[:800]})
                if E11_SERVER is not None:
                    E11_SERVER.kill()
            (OUT / "e13_loadprobe.json").write_text(json.dumps(E13_PROBE, indent=1, default=str), encoding="utf-8")
        # RC1 audits H1: the CPU reference after every GPU arm, with its own capped deadline; the GPU evidence is on disk
        # already (e13_loadprobe.json is written after each arm), so a truncated reference loses nothing
        for _tag, _tdir, _md in E13_RC1_PENDING:
            _arm = next((a for a in E13_PROBE["arms"] if a.get("tag") == _tag), {})
            _left = HARD_STOP - time.perf_counter() - 4 * 60
            _info["stage"] = f"rc1 reference {_tag}"
            if _left < 90:
                _arm["rc1_reference"] = {"skipped": f"{round(_left)} s left before the hard stop"}
            else:
                try:
                    _arm["rc1_reference"] = e13_rc1_reference(_tdir, _md, time.perf_counter() + min(15 * 60, _left))
                except Exception as _re:
                    _arm["rc1_reference"] = {"error": f"{type(_re).__name__}: {_re}"[:400]}
            print(f"E13 RC1 {_tag} reference:", json.dumps(_arm.get("rc1_reference"), default=str)[:3000], flush=True)
            (OUT / "e13_loadprobe.json").write_text(json.dumps(E13_PROBE, indent=1, default=str), encoding="utf-8")
        # rev 2 (F11): CPU-only, after the arms, its own guard (a timeout here never costs an arm)
        if globals().get("E13_VLLM_CHECK"):
            _info["stage"] = "vllm_registry"
            if HARD_STOP - time.perf_counter() < 6 * 60:
                E13_PROBE["vllm_registry"] = {"skipped": "less than 6 min left"}
            else:
                try:
                    E13_PROBE["vllm_registry"] = e13_vllm_registry()
                except Exception as _ve:
                    E13_PROBE["vllm_registry"] = {"error": f"{type(_ve).__name__}: {_ve}"[:400]}
            print("E13 vLLM 0.32 registry:", json.dumps(E13_PROBE["vllm_registry"], default=str)[:1500], flush=True)
            (OUT / "e13_loadprobe.json").write_text(json.dumps(E13_PROBE, indent=1, default=str), encoding="utf-8")
        _info["stage"] = "probe done"
    except Exception as _e:
        ENGINE_INFO["e13"]["error"] = f"{type(_e).__name__}: {_e}"[:3000]
        print("E13 LOADPROBE stopped:", ENGINE_INFO["e13"]["stage"], "|", ENGINE_INFO["e13"]["error"][:1500], flush=True)
        if E11_SERVER is not None:
            E11_SERVER.kill()
ENGINE_READY = False; CONCURRENCY = 0                 # load-only: the Duck never runs; CELL 0's fallback is the submission
ENGINE_INFO["fatal"] = "LOADPROBE: load-only notebook (no Duck)"
(OUT / "e13_loadprobe.json").write_text(json.dumps(E13_PROBE, indent=1, default=str), encoding="utf-8")
ENGINE_INFO["load_sec"] = round(time.perf_counter() - _t_engine, 1)
ENGINE_INFO["concurrency"] = 0
print("E13 LOADPROBE:", json.dumps([{k: a.get(k) for k in ("tag", "outcome", "sec")} for a in E13_PROBE["arms"]]), flush=True)


In [ ]:
# ==================================================================== CELL 6 — E12: Duck REPL agent, every task, E12_BRANCHES branches
# Every branch is admitted at once; Duck's priority gate keeps ARC3_MAX_ACTIVE_STREAMS conversations active (first
# branches in ORDER2 order before any second branch). A branch ends when the host verifies a submitted program on every
# train pair, at the deadline, or at the optional token cap. Full thinking: Duck's transcript.txt per branch.
_E12_INDEX = {t: i for i, t in enumerate(TASK_IDS)}


def _e12_order2_score(tid):   # = S49 ORDER2 ([>=3 train] + [1 test]) with the cells tie-break, sorted-id order last
    _t = CHALLENGES[tid]
    return int(len(_t["train"]) >= 3) + int(len(_t["test"]) == 1)


def _e12_total_cells(tid):
    _t = CHALLENGES[tid]
    _c = lambda g: len(g) * len(g[0]) if g and g[0] else 0
    return sum(_c(p["input"]) + _c(p["output"]) for p in _t["train"]) + sum(_c(p["input"]) for p in _t["test"])


E12_ORDER = sorted(TASK_IDS, key=lambda t: (-_e12_order2_score(t), _e12_total_cells(t), _E12_INDEX[t]))
E12_RECORDS = {t: [] for t in TASK_IDS}
E12_RUN = {"started_sec": None, "model_id": None, "streams": None, "deadline_in_sec": None, "error": None,
           "branches_done": 0, "branches_accepted": 0, "finished_sec": None}
_E12_LOCK = threading.Lock()
E12_OUT = OUT / "duck"
E12_OUT.mkdir(parents=True, exist_ok=True)


import tempfile as _tempfile
_E12_WRITE_LOCK = threading.Lock()   # rev 2 (GPT audits, reproduced): branches finish concurrently


def _e12_write_json_atomic(path, obj):
    """A unique temp file per write + os.replace (CELL 0's helper reuses one '<name>.tmp' for every writer)."""
    fd, tmp = _tempfile.mkstemp(prefix=Path(path).name + ".", suffix=".tmp", dir=str(Path(path).parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f)
        os.replace(tmp, path)
    except BaseException:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise


def e12_write_submission():
    with _E12_WRITE_LOCK:                          # one writer at a time; the snapshot is taken inside, so it is fresh
        sub = identity_submission(test_challenges)
        if RERUN:
            with _E12_LOCK:
                recs = {t: list(v) for t, v in E12_RECORDS.items() if v}
            for t, rows in E12A.build_submission({t: CHALLENGES[t] for t in recs}, recs).items():
                if t in sub and len(sub[t]) == len(rows):
                    sub[t] = rows
        _e12_write_json_atomic(SUBMISSION_PATH, sub)   # evaluation: keeps the competition-format test fallback


def _e12_branch_done(rec):
    with _E12_LOCK:
        E12_RECORDS.setdefault(rec["task_id"], []).append(rec)
        E12_RUN["branches_done"] += 1
        E12_RUN["branches_accepted"] += int(bool(rec.get("accepted")))
    try:
        e12_write_submission()
    except Exception as _e:
        print("E12 submission write failed:", repr(_e)[:200], flush=True)


_E12_STOP = threading.Event()


def _e12_heartbeat(every=300):
    while not _E12_STOP.wait(every):
        snap = {r.get("name"): r.get("value") for r in metrics_snapshot() if isinstance(r, dict) and "name" in r}
        keep = {k: v for k, v in snap.items() if any(s in k for s in ("num_running_reqs", "num_queue_reqs", "gen_throughput",
                                                                        "token_usage", "cache_hit_rate"))}
        print(f"E12 [{(time.perf_counter() - NOTEBOOK_T0) / 60:.0f} min] branches done {E12_RUN['branches_done']}"
              f"/{len(TASK_IDS) * E12_BRANCHES}, accepted {E12_RUN['branches_accepted']} | {json.dumps(keep)[:400]}", flush=True)


if ENGINE_READY and E10_SANDBOX_SELFTEST.get("ok"):
    try:
        try:
            E12_RUN["model_id"] = json.loads(_urlreq.urlopen(SGL_URL + "/v1/models", timeout=30).read())["data"][0]["id"]
        except Exception as _e:
            E12_RUN["model_id"] = str(SGL_MODEL_PATH)
            print("E12: /v1/models failed, using the model path:", repr(_e)[:200], flush=True)
        E12_RUN["streams"] = min(int(CONCURRENCY), E12_STREAMS or int(CONCURRENCY))
        os.environ["ARC3_MAX_ACTIVE_STREAMS"] = str(E12_RUN["streams"])
        _end = HARD_STOP - E12_STOP_BEFORE_HARD_SEC
        if E12_SOLVE_WINDOW_SEC:
            _end = min(_end, time.perf_counter() + E12_SOLVE_WINDOW_SEC)
        E12_RUN["deadline_in_sec"] = round(_end - time.perf_counter())
        E12_RUN["started_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
        print("E12 solving:", len(E12_ORDER), "tasks x", E12_BRANCHES, "branches | streams", E12_RUN["streams"],
              "| model", E12_RUN["model_id"], "| stops in", E12_RUN["deadline_in_sec"], "s | order head", E12_ORDER[:5], flush=True)
        threading.Thread(target=_e12_heartbeat, daemon=True).start()
        E12A.solve_all({t: CHALLENGES[t] for t in E12_ORDER}, E12_ORDER, out_dir=E12_OUT,
                       deadline=time.monotonic() + max(0.0, _end - time.perf_counter()), base_url=SGL_URL + "/v1",
                       model_id=E12_RUN["model_id"], branches=E12_BRANCHES, max_generated_tokens=E12_MAX_GEN_PER_BRANCH,
                       save_request_logs=E12_REQUEST_LOGS, on_branch_done=_e12_branch_done)
    except BaseException:
        E12_RUN["error"] = traceback.format_exc()[-3000:]
        print("E12 solver error (submission.json keeps what was written):", E12_RUN["error"], flush=True)
else:
    E12_RUN["error"] = f"not run: ENGINE_READY={ENGINE_READY}, sandbox self-test={E10_SANDBOX_SELFTEST.get('ok')}"
    print("E12:", E12_RUN["error"], "-> CELL 0 fallback submission", flush=True)
_E12_STOP.set()
E12_RUN["finished_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
try:
    e12_write_submission()
except Exception:
    print("E12 final submission write failed:", traceback.format_exc()[-1000:], flush=True)


In [ ]:
# ==================================================================== CELL 7 — E12: report, score (evaluation) and bundle
try:
    with _E12_LOCK:
        _recs = {t: list(v) for t, v in E12_RECORDS.items()}
    _gen = E12A.build_submission({t: CHALLENGES[t] for t in TASK_IDS}, _recs)
    write_json_atomic(OUT / "e12_generated_attempts.json", _gen)

    def _usage_summary(u):
        p = sum(int(x.get("prompt") or 0) for x in u); c = sum(int(x.get("cached") or 0) for x in u)
        return {"requests": len(u), "prompt_tokens": p, "cached_tokens": c, "cached_pct": round(100 * c / p, 1) if p else None,
                "completion_tokens": sum(int(x.get("completion") or 0) for x in u),
                "cached_pct_by_request": [round(100 * int(x.get("cached") or 0) / int(x["prompt"]), 1) if x.get("prompt") else None
                                          for x in u][:400]}
    branches = []
    for t in TASK_IDS:
        for r in _recs.get(t, []):
            subs = r.get("submissions") or []
            branches.append({"task": t, "branch": r.get("branch"), "accepted": bool(r.get("accepted")), "turns": r.get("turns"),
                             "failures": r.get("failures"), "note": r.get("note"), "generated_tokens": r.get("generated_tokens"),
                             "total_tokens": r.get("total_tokens"), "sec": round((r.get("t1") or 0) - (r.get("t0") or 0), 1),
                             "submissions": len(subs), "train_exact_per_submission": [sum(map(bool, s.get("train_exact") or []))
                                                                                       for s in subs],
                             "usage": _usage_summary(r.get("usage") or [])})
    _all_u = [x for t in TASK_IDS for r in _recs.get(t, []) for x in (r.get("usage") or [])]
    report = {"run_id": RUN_ID, "profile": PROFILE, "rerun": RERUN, "configuration": CONFIG, "engine": ENGINE_INFO,
              "e12_run": E12_RUN, "order": E12_ORDER, "branches": branches, "usage_all": {k: v for k, v in _usage_summary(_all_u).items()
                                                                                        if k != "cached_pct_by_request"},
              "tasks_planned": len(TASK_IDS), "queries_planned": sum(len(CHALLENGES[t]["test"]) for t in TASK_IDS)}
    if SOLUTIONS is not None:
        hits = pass1 = pass2 = total = 0; task_norm = 0.0; recs_eval = []
        for t in TASK_IDS:
            th = 0
            for qi, att in enumerate(_gen[t]):
                gold = SOLUTIONS[t][qi]; a = att["attempt_1"] == gold; b = att["attempt_2"] == gold
                hits += a or b; pass1 += a; pass2 += b; total += 1; th += a or b
                acc = [s for r in _recs.get(t, []) for s in (r.get("submissions") or []) if s.get("accepted")]
                recs_eval.append({"task": t, "query": qi, "hit": bool(a or b), "pass1": bool(a), "pass2": bool(b),
                                  "accepted_programs": len(acc),
                                  "accepted_programs_correct": sum(1 for s in acc if qi < len(s.get("test_outputs") or [])
                                                                   and s["test_outputs"][qi] == gold)})
            task_norm += th / max(1, len(_gen[t]))
        report.update(eval_query_hits=hits, eval_query_total=total, eval_pass1_hits=pass1, eval_pass2_hits=pass2,
                      eval_task_norm_pct=round(100 * task_norm / max(1, len(TASK_IDS)), 2), eval_unit_records=recs_eval)
    try:
        report["e11"] = {"decode_summary": e11_decode_summary(),
                         "shim": engine.stats if isinstance(getattr(engine, "stats", None), dict) else None,
                         "server_log": str(E11_SERVER.log_path) if E11_SERVER is not None else None}
    except Exception as _e11e:
        report["e11"] = {"error": repr(_e11e)[:300]}
    write_json_atomic(OUT / "e12_run_report.json", report)
    print("E12 RUN REPORT:", OUT / "e12_run_report.json", "| eval", report.get("eval_query_hits"), "/", report.get("eval_query_total"),
          "| task-norm", report.get("eval_task_norm_pct"), "| accepted branches", E12_RUN["branches_accepted"],
          "| cached", report["usage_all"].get("cached_pct"), "%", flush=True)
    e12_write_submission()
    _pack = WORKING / "Q27X_E12_DUCK_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in (SUBMISSION_PATH, OUT / "e12_run_report.json", OUT / "e12_generated_attempts.json",
                   OUT / "sglang_decode_summary.json", OUT / "sglang_requests.jsonl"):
            if fp.exists():
                z.write(fp, arcname=fp.name)
        for fp in sorted(E12_OUT.rglob("*")):     # transcripts (full thinking) + branch results
            if fp.is_file() and (fp.name in ("transcript.txt", "branch_result.json") or fp.name.endswith("requests.jsonl")):
                z.write(fp, arcname="duck/" + fp.relative_to(E12_OUT).as_posix())
        if E11_SERVER is not None and E11_SERVER.log_path.exists():
            z.write(E11_SERVER.log_path, arcname=E11_SERVER.log_path.name)
    print("E12 bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
except BaseException:
    print("E12 reporting error (submission.json still exists):", traceback.format_exc()[-3000:], flush=True)
try:
    e11_kill_server(wait=False)
except Exception:
    pass
